# 6e · E2E relation canonicalization  (twin of nb7b, for the single-call graph)

Runs the **same** predicate canonicalization as nb7b, but on the **e2e** graph. Reads
`results/<id>/e2e/nary_graph_typed_consolidated.json` (from **6d**, entity types already canonicalized), matches each
free-form `relation_phrase` to a schema.org **or** vocabulary property (else mints one), and writes
`results/<id>/e2e/nary_graph_extended.json` + `e2e/nary_relation_matching_review.json`.

**Separate vocabulary:** grows `results/_global/e2e/nary_relation_vocabulary.json` — independent of the
staged track's `nary_relation_vocabulary.json`, so the two methods' predicate economies stay comparable.

Run **after 6d**. The optional global re-consolidation / property-hierarchy passes from nb7 port over
unchanged (operate on the vocab + ext files) — add them here if/when you want that cleanup.

In [12]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [13]:
# ---- WHICH MODEL'S RESULTS ---------------------------------------------------------------
# Everything is read from and written to results/<RESULTS_MODEL>/. Choose any subfolder that
# exists there: "gemini-2.5-flash", "qwen3-14b", "qwen3-32b", "llama33-70b", ... (the names are
# the BACKENDS keys in notebooks/llm_backend.py).
RESULTS_MODEL = "gemini-2.5-flash"

# Which model runs THIS notebook's LLM calls. Usually the same one; set it to
# "gemini-2.5-flash" to process another model's outputs with Gemini (e.g. as an eval judge).
# Anything else needs a vLLM server + tunnel — see cluster/README.md.
BACKEND = RESULTS_MODEL
# ------------------------------------------------------------------------------------------

# --- Config: n-ary relation canonicalization (nb7b) ----------------------------------------------------
import json, re, os, copy
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

SCHEMA_DIR  = ROOT / "schema_org_files"     # property pool from nb5
RESULTS_DIR = ROOT / "results" / RESULTS_MODEL   # reads e2e/nary_graph_typed_consolidated.json (6d)
# Canonicalize the SAME variant as 6d. VARIANT="" -> default; else nary_graph_<VARIANT>_typed_consolidated.json.
VARIANT = "dedup"                                  # MUST match 6d's VARIANT
_sfx    = f"_{VARIANT}" if VARIANT else ""
# ---- RERANKER (cross-encoder second stage) ------------------------------------------------
# e5 fetches RERANK_FROM candidates, then a cross-encoder — which reads the query and the
# candidate TOGETHER, unlike the bi-encoder — cuts them down to TOP_K_MATCH. Turning this on
# changes the REAL build, so its outputs get their own filenames (see _osfx below) and never
# overwrite the plain-retrieval baseline.
RERANK        = True
RERANK_FROM   = 50     # how deep to retrieve before reranking. This is the reranker's ceiling:
                        # it can only promote what e5 already fetched. Deeper = better recall but
                        # linearly more cross-encoder work. Keep it the SAME in 6d and 6e.
RERANK_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
# 22M params, English, CPU-friendly. Stronger but much slower on CPU: "BAAI/bge-reranker-base"
# (278M) or "BAAI/bge-reranker-v2-m3" (568M, multilingual).

# _sfx  identifies the VARIANT and is used to READ what the previous notebook wrote.
# _osfx adds the reranker tag and is used for everything THIS notebook WRITES, so a reranked run
# sits beside the baseline instead of clobbering it. 6e reads 6d's output, so it uses _osfx for
# that read too — which means 6d and 6e must be run with the SAME RERANK setting.
_osfx = _sfx + ("_rr" if RERANK else "")

VOCAB_PATH  = RESULTS_DIR / f"_global/e2e/nary_relation_vocabulary{_osfx}.json"   # SEPARATE per variant

MATCH_MODEL  = "intfloat/e5-base-v2"       # ONE embedding model for the whole vocabulary (don't mix spaces)
GEMINI_MODEL = "gemini-2.5-flash"
EXPAND_MODE  = True                        # True: admit minted properties (grow). False: restrict to schema.org u vocab.

# No mint-time canonicalization: a minted property is admitted as-is during the incremental build.
# Collapsing synonymous / broader-narrower mints is deferred to the OPTIONAL global re-consolidation
# pass at the bottom (consolidate_vocab) — run it to clean up, or skip it to keep the raw vocabulary.

TOP_K_MATCH  = 10                          # candidate properties shown to the selection LLM (Hook A)

# ---- RETRIEVAL VARIANT (multi-anchor indexing) --------------------------------------------
# How each property is represented in the document pool. See the anchor-pool cell for the why.
#   "original" : ONE vector per property   "about: A subject matter relation … Also called concerning, …"
#   "A"        : + bare surface anchors    'about' · 'concerning' · 'regarding'
#   "B"        : + surface + sense tag     'about: subject matter relation' · 'concerning: subject matter relation'
#   "C"        : SURFACE FORMS ONLY, on BOTH sides — no definitions anywhere. The query becomes the
#                bare relation phrase ('caused by'), the documents the bare label and synonyms
#                ('cause', 'causal relation'). 192 of 213 query definitions open with the word
#                "Indicates" and average 9.6 words, so most of a query's mass is boilerplate shared
#                with every other query — which is a better explanation of the flat 0.83 score band
#                than anything on the document side. C removes it from both sides. camelCase is
#                split so 'causedBy' can meet the synonym 'caused by' (128 of 193 phrases are
#                camelCase). C keeps NO blended vector: it is a clean test, not a safety net.
# A property's score is the MAX over its anchors, so the rest of the pipeline is unchanged: it
# still sees one score per property. "original" reproduces the current behaviour exactly.
ANCHOR_MODE  = "original"
# Tiebreak penalty SUBTRACTED from synonym anchors, so a synonym must win clearly to outrank a
# property's own label. It must be ADDITIVE: e5 cosine similarities are squeezed into ~0.75-0.96,
# so a multiplicative 0.9 would demand a raw similarity of ~0.94 — the top of the whole
# distribution — and would silently disqualify every synonym anchor. The typical rank1-rank2 gap
# is ~0.010, so 0.005 is half a place. Set 0.0 for no penalty at all.
ANCHOR_PENALTY = 0.005
DROP_AMBIGUOUS_SYNONYMS = True   # skip synonyms claimed by >1 property ('issued by', 'belongs to')
ALIGN_QUERY_STYLE = True                   # append "Also called <synonyms>." to each relation-phrase query

# Which articles nb7 runs on — and the order the vocabulary grows along (order matters: incremental).
#   None  -> every article that has a nary_graph.json (from nb6b), sorted.
#   list  -> exactly these ids, in this order, e.g. ["art2", "art1", "art20"].
#SELECTED_ARTICLES = None
SELECTED_ARTICLES = ["art1", "art2", "art4", "art5"]

_available = sorted(p.relative_to(RESULTS_DIR).parts[0] for p in RESULTS_DIR.glob(f"*/e2e/nary_graph{_osfx}_typed_consolidated.json"))
if SELECTED_ARTICLES is None:
    ARTICLE_ORDER = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print("WARNING: skipping (no e2e/nary_graph_typed_consolidated.json):", missing)
    ARTICLE_ORDER = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_ORDER, "No runnable articles selected (need 6d e2e/nary_graph_typed_consolidated.json). Check SELECTED_ARTICLES."
print("expand:", EXPAND_MODE, "| match:", MATCH_MODEL, "| articles:", ARTICLE_ORDER)

expand: True | match: intfloat/e5-base-v2 | articles: ['art1', 'art2', 'art4', 'art5']


In [14]:
# First run only (sentence-transformers pulls torch; google-genai for the LLM stages).
# %pip install -q sentence-transformers google-genai numpy json-repair

In [15]:
# Embedding model (loaded once). e5 needs "query: " on the query side and "passage: " on documents;
# both the relation-phrase queries and every document (schema.org OR vocab property) go through the
# same convention — identical to nb4.
from sentence_transformers import SentenceTransformer
_MODEL = SentenceTransformer(MATCH_MODEL)

def _prefixes(name):
    n = name.lower()
    if "e5" in n:  return "query: ", "passage: "
    if "bge" in n: return "Represent this sentence for searching relevant passages: ", ""
    return "", ""
QP, DP = _prefixes(MATCH_MODEL)

def embed(texts, prefix):
    return np.asarray(_MODEL.encode([prefix + t for t in texts],
                                    normalize_embeddings=True, batch_size=64))
print("embedder ready:", MATCH_MODEL, "| query_prefix", repr(QP), "| doc_prefix", repr(DP))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedder ready: intfloat/e5-base-v2 | query_prefix 'query: ' | doc_prefix 'passage: '


In [16]:
# Schema.org PROPERTY document pool (fixed) — embedded once (from nb5). Vocabulary properties are
# embedded per article inside retrieval, since the vocabulary grows during the run.
schema_pool   = json.loads((SCHEMA_DIR / "schema_properties_enriched.json").read_text())
SCHEMA_LABELS = {p["label"] for p in schema_pool}
schema_emb    = embed([p["embedding_text"] for p in schema_pool], DP)
print(len(schema_pool), "schema.org properties embedded")

1521 schema.org properties embedded


In [17]:
# --- Multi-anchor document pool (retrieval variants A / B) -------------------------------------
# Today every property is ONE vector, mean-pooled over "label: definition. Also called syn1, syn2…".
# Mean pooling dilutes the surface forms — the synonyms are in the text but contribute ~1/35 of the
# average — which is why candidates arrive in a flat score band (0.791-0.817 in art1) and the right
# property can sit at rank 7. A/B index each property SEVERAL times, once per surface form, and score
# it by the MAX over its anchors, so an OIE phrase only has to be close to ONE of them:
#
#   original : the blended vector, exactly as today
#   A        : blended + bare surface forms          'about' · 'concerning' · 'on the topic of'
#   B        : blended + surface + short sense tag   'about: subject matter relation' · …
#   C        : surface forms ONLY — 'about' · 'concerning' — and the QUERY drops its definition too
#
# A spreads the anchors furthest but loses the sense ('record' would pull weather queries towards
# `album`); B keeps a ~1:1 surface-to-context ratio, enough to pin the sense without swamping the
# surface form. Every mode KEEPS the blended vector, so the max can only match or beat today.
# Synonym anchors are docked ANCHOR_PENALTY (ADDITIVE — see the config cell for why not a factor).
def _surface(s):
    """camelCase/PascalCase -> spaced lowercase, so the phrase 'causedBy' can meet the synonym
    'caused by'. 128 of the 193 OIE phrases are camelCase, as is every schema.org label; the
    synonyms are already natural language and pass through unchanged."""
    s = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", " ", s or "")
    s = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", " ", s)          # HTTPServer -> HTTP Server
    return re.sub(r"[^0-9a-zA-Z]+", " ", s).strip().lower()

_SYN_RE  = re.compile(r"Also called ([^.]+)\.")
_EXAM_RE = re.compile(r"\s*Examples?:.*$", re.S)

def _gloss(text):
    """The definition itself — 'Also called …' and 'Examples: …' stripped off."""
    return _EXAM_RE.sub("", _SYN_RE.sub("", text or "")).strip()

def _synonyms(text):
    m = _SYN_RE.search(text or "")
    return [s.strip().strip('"').strip() for s in m.group(1).split(",") if s.strip()] if m else []

def _sense_tag(gloss, label):
    """Variant B's short sense hint. nb5's enrichment prompt mandates 'A <X> relation between …',
    so the head before ' between ' is a 2-4 word gloss of the relation TYPE (holds for ~64% of
    properties); otherwise fall back to the first clause, then to the label."""
    if not gloss:
        return label
    head = gloss.split(" between ")[0] if " between " in gloss else re.split(r"[,.;]", gloss)[0]
    head = re.sub(r"^(an?|the)\s+", "", head.strip(), flags=re.I).strip()
    return head[:60] or label

# A synonym anchor is only useful if it points at ONE property. Two ways it fails to:
#   * claimed by several properties  — 'issued by' belongs to BOTH publisher and recordLabel,
#     'belongs to' to ten. As an anchor it drags every query towards all its owners (6g's failure).
#   * it IS another property's label — 'value' is listed as a synonym of `amount`, but schema.org
#     has an actual property called `value`, which should own that anchor outright (35 such cases).
# Both are dropped; a property's OWN label is of course always kept as its anchor.
_syn_owner = defaultdict(set)
for _p in schema_pool:
    for _s in _synonyms(_p.get("enriched_comment") or ""):
        _syn_owner[_s.lower()].add(_p["label"])
AMBIGUOUS_SYNONYMS = {s for s, owners in _syn_owner.items() if len(owners) > 1}
SCHEMA_LABEL_BY_LOWER = {p["label"].lower(): p["label"] for p in schema_pool}

def _usable_synonyms(syns, own_label):
    """Synonyms that can serve as anchors for the property labelled `own_label`."""
    if not DROP_AMBIGUOUS_SYNONYMS:
        return syns
    out = []
    for s in syns:
        k = s.lower()
        if k in AMBIGUOUS_SYNONYMS:                       # several properties claim it
            continue
        if SCHEMA_LABEL_BY_LOWER.get(k, own_label) != own_label:   # it is another property's label
            continue
        out.append(s)
    return out

def _anchors_for(item, kind, mode):
    """-> [(text, weight)] : the vectors this ONE property contributes to the pool."""
    if kind == "schema.org":
        label   = item["label"]
        blended = item.get("embedding_text") or f'{label}: {item.get("comment", "")}'
        rich    = item.get("enriched_comment") or item.get("comment") or ""
        syns    = _synonyms(rich)
    else:                                              # a grown-vocabulary property
        label   = item["canonical_label"]
        blended = vocab_embedding_text(item)
        rich    = item.get("definition") or ""
        syns    = list(item.get("aliases") or [])

    if mode == "original":
        out = [(blended, 0.0)]
    elif mode == "C":                                  # surface only — the blended vector is NOT kept
        out = [(_surface(label), 0.0)]
        out += [(_surface(s), -ANCHOR_PENALTY) for s in _usable_synonyms(syns, label)]
    else:
        out = [(blended, 0.0)]
        syns = _usable_synonyms(syns, label)
        if mode == "A":
            out.append((label, 0.0))
            out += [(s, -ANCHOR_PENALTY) for s in syns]
        elif mode == "B":
            tag = _sense_tag(_gloss(rich), label)
            out.append((f"{label}: {tag}", 0.0))
            out += [(f"{s}: {tag}", -ANCHOR_PENALTY) for s in syns]
        else:
            raise ValueError(f"unknown ANCHOR_MODE {mode!r} — use 'original', 'A', 'B' or 'C'")

    best = {}                                          # dedupe, keeping the smallest penalty
    for t, w in out:
        t = (t or "").strip()
        if t:
            best[t] = max(w, best.get(t, -9.0))
    return list(best.items()) or [(_surface(label) if mode == "C" else label, 0.0)]   # never zero anchors

def build_anchor_pool(items, kind, mode):
    """Embed every item's anchors into one matrix, remembering which rows belong to which item.
    Anchors are laid out CONTIGUOUSLY per item so the group-max is a single np.maximum.reduceat."""
    texts, offsets, starts = [], [], []
    for it in items:
        starts.append(len(texts))
        for t, w in _anchors_for(it, kind, mode):
            texts.append(t); offsets.append(w)
    starts = np.asarray(starts)
    return {"texts": texts, "starts": starts,
            "ends": np.append(starts[1:], len(texts)),
            "offsets": np.asarray(offsets, dtype="float32"),
            "emb": embed(texts, DP), "kind": kind, "mode": mode}

_ANCHOR_POOLS = {}

def schema_anchor_pool(mode):
    """Cached anchor pool over schema_pool. 'original' reuses the vectors cell 5 already embedded,
    so switching back to it costs nothing.

    The cache key includes ANCHOR_PENALTY and DROP_AMBIGUOUS_SYNONYMS because both are baked into
    the pool when it is built. Keying on `mode` alone would silently serve a stale pool after you
    changed a knob and re-ran only the comparison cell — the run would look fine and be wrong."""
    key = (mode, ANCHOR_PENALTY, DROP_AMBIGUOUS_SYNONYMS)
    if key not in _ANCHOR_POOLS:
        if mode == "original":
            n = len(schema_pool)
            _ANCHOR_POOLS[key] = {"texts": [p["embedding_text"] for p in schema_pool],
                                  "starts": np.arange(n), "ends": np.arange(1, n + 1),
                                  "offsets": np.zeros(n, dtype="float32"),
                                  "emb": schema_emb, "kind": "schema.org", "mode": mode}
        else:
            _ANCHOR_POOLS[key] = build_anchor_pool(schema_pool, "schema.org", mode)
        p = _ANCHOR_POOLS[key]
        print(f"  anchor pool [{mode}]: {len(p['texts'])} vectors for {len(schema_pool)} properties"
              f"  ({len(p['texts']) / len(schema_pool):.1f} per property)")
    return _ANCHOR_POOLS[key]

def anchor_scores(q_emb, pool):
    """-> (best, raw). best[q, item] = max over that item's (weighted) anchors — one score per
    property, exactly the contract the rest of the notebook already expects. `raw` is per-anchor,
    kept so the winning anchor can be named on the candidate record."""
    raw  = (q_emb @ pool["emb"].T) + pool["offsets"]
    best = np.maximum.reduceat(raw, pool["starts"], axis=1)
    return best, raw

def _winning_anchor(pool, raw_row, item_idx):
    """Which anchor text actually produced this property's score (None when it has only one)."""
    a, b = int(pool["starts"][item_idx]), int(pool["ends"][item_idx])
    if b - a <= 1:
        return None
    return pool["texts"][a + int(np.argmax(raw_row[a:b]))]

print(f"anchors ready | mode={ANCHOR_MODE} synonym_penalty={ANCHOR_PENALTY} "
      f"| {len(AMBIGUOUS_SYNONYMS)} ambiguous synonym(s) "
      f"{'dropped' if DROP_AMBIGUOUS_SYNONYMS else 'KEPT'}")


anchors ready | mode=original synonym_penalty=0.005 | 166 ambiguous synonym(s) dropped


In [18]:
# The growing property vocabulary store. One entry per canonical minted property:
#   id, canonical_label (camelCase property name), definition, parent (subPropertyOf: a schema.org OR
#   vocab property label, or null), domain[] / range[] (subject/object type hints), aliases[] (its
#   synonyms), provenance[], support, first_seen, and path (derived: super-property chain, stamped on save).
def load_vocab():
    if VOCAB_PATH.exists():
        return json.loads(VOCAB_PATH.read_text())
    return {"properties": [], "next_id": 1}

def save_vocab(vocab):
    _annotate_vocab_paths(vocab)                 # refresh each entry's derived `path` from its parent chain
    VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

def _sig(domain, range_):
    """domain → range for HUMAN display / the LLM judge prompt (the '→' glyph is fine there). Kept
    OUT of embedding_text — see vocab_embedding_text."""
    d = "/".join(domain) if domain else "ANY"
    r = "/".join(range_) if range_ else "ANY"
    return f"{d} → {r}"

def vocab_embedding_text(t):
    # "label: definition Also called <aliases>." — the SAME style as nb7's relation-phrase query and
    # nb5's schema property docs, so query and both document sources share one embedding style. The
    # domain -> range signature is intentionally not embedded (see nb5); it goes to the LLM judge only.
    also = f" Also called {', '.join(t['aliases'])}." if t.get("aliases") else ""
    return f'{t["canonical_label"]}: {t.get("definition", "")}{also}'

def _find_vocab_by_label(vocab, label):
    for t in vocab["properties"]:
        if t["canonical_label"] == label or label in t.get("aliases", []):
            return t
    return None

def _is_vocab_label(vocab, label):
    return _find_vocab_by_label(vocab, label) is not None

def _bump(ent, aid, phrase, instances, flip=False):
    # provenance instances are oriented to the PROPERTY's direction (swapped when flip=True), so they
    # read the same way as the stored triples.
    ent["support"] = ent.get("support", 0) + len(instances)
    def _pair(i):
        return f'{i["object_text"]} -> {i["subject_text"]}' if flip else f'{i["subject_text"]} -> {i["object_text"]}'
    ent.setdefault("provenance", []).append(
        {"article": aid, "relation_phrase": phrase, "direction": ("inverse" if flip else "as_is"),
         "instances": [_pair(i) for i in instances]})

In [19]:
# Hook A — retrieve top-k candidate properties for each relation phrase against schema.org u current
# vocabulary. Both sources expose their domain->range `signature` and their `subPropertyOf` parent, so
# the selection LLM can judge direction/domain-range fit and reach for either as a mint parent.
_SCHEMA_BY_LABEL = {p["label"]: p for p in schema_pool}

def _schema_super(label):
    """Immediate super-property (subPropertyOf) of a schema.org property, or None. schema.org lists
    sub_property_of as a (usually empty) list; we take the first schema.org super-property."""
    p = _SCHEMA_BY_LABEL.get(label)
    for sup in (p.get("sub_property_of") if p else []) or []:
        if sup in SCHEMA_LABELS:
            return sup
    return None

def _schema_super_chain(label):
    """Top-down super-property chain INCLUDING `label`: [root_super, ..., label]. Sparse for
    properties (most schema.org properties have no super-property), so usually just [label]."""
    chain, cur, seen = [], label, set()
    while cur and cur not in seen:
        seen.add(cur); chain.append(cur); cur = _schema_super(cur)
    return list(reversed(chain))

def vocab_ancestor_path(t, vocab):
    """super-property chain for a vocab property (top-down, ending at the immediate parent, EXCLUDING
    the property itself). Climbs `parent` (subPropertyOf) links up through the vocabulary, then splices
    in the schema.org super-chain of the first schema property it reaches. A `seen` guard prevents loops."""
    up, cur, seen = [], t.get("parent"), set()
    while cur and cur not in seen:
        seen.add(cur)
        ent = _find_vocab_by_label(vocab, cur)
        if ent:                                   # another vocab property -> keep climbing the vocab tree
            up.append(cur); cur = ent.get("parent")
        else:                                     # a schema.org property -> splice its super-chain in
            up.extend(reversed(_schema_super_chain(cur))); cur = None
    return " > ".join(reversed(up))

def _annotate_vocab_paths(vocab):
    """Stamp each entry's derived `path` (super-property chain, ending at the immediate parent).
    Derived from the `parent` chains (the source of truth) — refreshed on every save."""
    for t in vocab["properties"]:
        t["path"] = vocab_ancestor_path(t, vocab)
    return vocab

def _full_path(tref, vocab):
    """Full super-property path for a resolved property reference (used in the review's chosen block)."""
    if tref.get("source") == "vocab" and tref.get("vocab_id"):
        ent = next((t for t in vocab["properties"] if t["id"] == tref["vocab_id"]), None)
        if ent is not None:
            return vocab_ancestor_path(ent, vocab)
    if tref.get("source") == "schema.org" and tref.get("label"):
        return " > ".join(_schema_super_chain(tref["label"])[:-1])   # super-properties only
    return None

def retrieve_candidates(phrase_texts, vocab, mode=None, top_k=None, phrases=None, rerank=None):
    """Top-k properties per relation phrase. Each property is scored by the MAX over its anchors
    (see the anchor-pool cell), so the records handed downstream are the same shape as before —
    one score per property — plus `matched_anchor`, the anchor text that produced that score.
    `mode` defaults to ANCHOR_MODE; pass it explicitly to compare variants without re-running.
    Mode C also replaces the QUERY with the bare relation phrase (no definition): pass `phrases`
    to give it the raw phrases, else they are recovered from `phrase_texts` ("phrase: definition").

    With `rerank` (default: the RERANK flag) it fetches RERANK_FROM deep and cross-encodes down to
    top_k, so turning RERANK on in the config changes the REAL build, not just the comparison."""
    mode   = ANCHOR_MODE if mode is None else mode
    top_k  = TOP_K_MATCH if top_k is None else top_k
    rerank = RERANK if rerank is None else rerank
    fetch  = max(top_k, RERANK_FROM) if rerank else top_k
    q_texts = phrase_texts
    if mode == "C":
        src = phrases if phrases is not None else [t.split(": ", 1)[0] for t in phrase_texts]
        q_texts = [_surface(p) for p in src]
    q_emb = embed(q_texts, QP)

    spool         = schema_anchor_pool(mode)
    sims, s_raw   = anchor_scores(q_emb, spool)
    meta          = [("schema.org", p) for p in schema_pool]
    n_schema      = len(schema_pool)
    vpool, v_raw  = None, None
    if vocab["properties"]:                      # the vocab grows during a build -> embedded per call
        vpool        = build_anchor_pool(vocab["properties"], "vocab", mode)
        vsims, v_raw = anchor_scores(q_emb, vpool)
        sims = np.hstack([sims, vsims])
        meta = meta + [("vocab", t) for t in vocab["properties"]]

    topk = np.argsort(-sims, axis=1)[:, :fetch]
    out = []
    for ti in range(len(phrase_texts)):
        row = []
        for j in topk[ti]:
            src, obj = meta[j]
            if src == "schema.org":
                row.append({"source": "schema.org", "label": obj["label"], "signature": obj.get("signature", ""),
                            "parent": _schema_super(obj["label"]), "inverse_of": obj.get("inverse_of"),
                            "score": round(float(sims[ti, j]), 3),
                            "matched_anchor": _winning_anchor(spool, s_raw[ti], j),
                            "comment": obj.get("enriched_comment") or obj.get("comment", "")})
            else:
                row.append({"source": "vocab", "id": obj["id"], "label": obj["canonical_label"],
                            "signature": _sig(obj.get("domain"), obj.get("range")), "parent": obj.get("parent"),
                            "inverse_of": None,
                            "score": round(float(sims[ti, j]), 3),
                            "matched_anchor": _winning_anchor(vpool, v_raw[ti], j - n_schema),
                            "comment": obj.get("definition", "")})
        out.append(row)
    if rerank:      # second stage: cross-encode the deep list down to the ten the LLM sees
        out = [rerank_candidates(q, row, top_k) for q, row in zip(phrase_texts, out)]
    return out

In [20]:
# --- Cross-encoder reranker (no training) -------------------------------------------------------
# The recall@K diagnostic says the bottleneck is SELECTION, not the embedding space: 74 of the 78
# reference properties are reachable within the top 100, but only ~60 survive the cut at 10. So the
# lever is choosing a better ten, not embedding differently.
#
# A bi-encoder (e5) scores a query and a document from two vectors built independently — it never
# sees them together, which is why `recordedAt` (audio-visual capture) can outrank `hasMeasurement`
# for a temperature query. A cross-encoder reads the pair JOINTLY and scores the match directly. It
# is far too slow to run over 1,521 properties per phrase, which is exactly why it goes second:
#
#     e5 retrieves RERANK_FROM (cheap, high recall)  ->  cross-encoder cuts to the ten the LLM sees
# RERANK / RERANK_FROM / RERANK_MODEL are set in the CONFIG CELL, with the other knobs —
# they must live there because _osfx (the reranked output tag) and VOCAB_PATH are built
# from RERANK. Defining them again here would silently override the config cell.
# 22M params, English, CPU-friendly (~10-35 s for this corpus). Stronger but much slower on CPU:
# "cross-encoder/ms-marco-MiniLM-L-6-v2" "BAAI/bge-reranker-base" (278M) or "BAAI/bge-reranker-v2-m3" (568M, multilingual).

_RERANKER = None

def _reranker():
    """Loaded on first use, so the notebook costs nothing while RERANK is off."""
    global _RERANKER
    if _RERANKER is None:
        from sentence_transformers import CrossEncoder
        print(f"loading reranker {RERANK_MODEL} ...")
        _RERANKER = CrossEncoder(RERANK_MODEL)
    return _RERANKER

def rerank_candidates(query, candidates, top_k):
    """Re-order ONE phrase's candidates by cross-encoder relevance, then cut to top_k.

    The document side is "label: definition" — the definition is what lets the model separate
    schema.org's idiosyncratic senses, and it is the half mode C threw away. `score` becomes the
    cross-encoder score (a logit, so it can be negative and is NOT comparable to a cosine); the
    retrieval score and rank are kept as `embed_score` / `embed_rank` so you can see what moved."""
    if not candidates:
        return []
    pairs  = [(query, f'{c["label"]}: {(c.get("comment") or "")[:400]}') for c in candidates]
    scores = np.asarray(_reranker().predict(pairs, batch_size=64, show_progress_bar=False),
                        dtype="float64")
    out = []
    for j in np.argsort(-scores)[:top_k]:
        c = dict(candidates[int(j)])
        c["embed_score"] = c["score"]
        c["embed_rank"]  = int(j) + 1
        c["score"]       = round(float(scores[int(j)]), 4)
        out.append(c)
    return out

# --- one entry point for both the comparison and the recall diagnostic --------------------------
# A "spec" is either an anchor mode ("original", "A", "B", "C") or "rerank:<mode>", meaning
# retrieve with <mode> then cross-encode down. Keeping it a spec rather than a fifth anchor mode
# matters because reranking is a SECOND STAGE, not another way to represent the pool.
def _col(spec):
    return spec.replace(":", "_")

def candidates_for(spec, qtexts, vocab, top_k, phrases=None):
    # rerank=False everywhere: the spec decides, never the global RERANK flag, so the comparison
    # means the same thing whether or not the build is currently configured to rerank.
    if not spec.startswith("rerank:"):
        return retrieve_candidates(qtexts, vocab, mode=spec, top_k=top_k, phrases=phrases,
                                   rerank=False)
    base = spec.split(":", 1)[1]
    deep = retrieve_candidates(qtexts, vocab, mode=base, top_k=max(top_k, RERANK_FROM),
                               phrases=phrases, rerank=False)
    # rerank against the NATURAL-LANGUAGE query (phrase + definition), never mode C's bare surface
    return [rerank_candidates(q, row, top_k) for q, row in zip(qtexts, deep)]

print(f"reranker ready (lazy) | RERANK={RERANK} | {RERANK_MODEL}")
print(f"  outputs -> *{_osfx}*   |  vocab: {VOCAB_PATH.name}")


reranker ready (lazy) | RERANK=True | cross-encoder/ms-marco-MiniLM-L-6-v2
  outputs -> *_dedup_rr*   |  vocab: nary_relation_vocabulary_dedup_rr.json


In [21]:
# Article inputs + Gemini client (JSON mode, temperature 0 for reproducibility).
def load_article(aid):
    """Read nb6b's nary_graph.json and group relation instances by their free-form relation_phrase.
    `by_id` resolves BOTH entities (E..) and n-ary nodes (EV../OBS../CL..) to a type, so the
    subject->object signatures work for n-ary-node endpoints too."""
    g = json.loads((RESULTS_DIR / aid / f"e2e/nary_graph{_osfx}_typed_consolidated.json").read_text())
    by_id = {e["id"]: {"id": e["id"], "text": e.get("text"), "type": e.get("type")}
             for e in g.get("entities", [])}
    for n in g.get("nary_nodes", []):                      # n-ary nodes are first-class endpoints
        by_id[n["id"]] = {"id": n["id"], "text": n.get("label"),
                          "type": n.get("type") or n.get("node_kind")}
    rinfo   = {d["relation_phrase"]: {"definition": d.get("definition")}
               for d in g.get("relation_definitions", [])}
    phrases = list(dict.fromkeys(r["relation_phrase"] for r in g["relations"]))
    def qtext(ph):
        info = rinfo.get(ph, {})
        q = f'{ph}: {info.get("definition", "")}'.strip()
        if ALIGN_QUERY_STYLE and info.get("synonyms"):     # nary graph has no synonyms -> no-op, kept for parity
            q = f'{q} Also called {", ".join(info["synonyms"])}.'
        return q
    groups = {}
    for r in g["relations"]:
        groups.setdefault(r["relation_phrase"], []).append(r)
    return g, by_id, rinfo, phrases, [qtext(p) for p in phrases], groups

def _etype(by_id, eid):
    return (by_id.get(eid) or {}).get("type")

def group_signatures(instances, by_id):
    """Distinct observed subject_type -> object_type signatures within a relation-phrase group."""
    sigs = Counter(f'{_etype(by_id, i["subject_id"])} → {_etype(by_id, i["object_id"])}' for i in instances)
    return [s for s, _ in sigs.most_common()]

from google import genai
from google.genai import types as genai_types
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in the env and restart the kernel."
client = genai.Client(api_key=GEMINI_API_KEY)

def gemini_json(system, user):
    """One JSON-mode call. Thinking OFF so the WHOLE output budget goes to the JSON answer — gemini-2.5-flash
    dynamic thinking otherwise eats the budget and truncates the JSON mid-string on big articles."""
    cfg = dict(system_instruction=system, temperature=0, response_mime_type="application/json")
    if hasattr(genai_types, "ThinkingConfig"):
        cfg["thinking_config"] = genai_types.ThinkingConfig(thinking_budget=-1)
    import time
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=GEMINI_MODEL, contents=user, config=genai_types.GenerateContentConfig(**cfg))
    global LAST_USAGE          # this call's tokens — now INCLUDING thinking tokens — and its latency
    LAST_USAGE = nb_parallel.record_gemini_usage(resp, time.perf_counter() - t0)
    return resp

def parse_json(resp):
    """Parse Gemini JSON from a response (or raw text), tolerating fences / trailing commas, with a
    CLEAR message on truncation instead of a cryptic JSONDecodeError."""
    text = resp if isinstance(resp, str) else (getattr(resp, "text", "") or "")
    s = text.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\s*", "", s); s = re.sub(r"\s*```$", "", s).strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        pass
    s2 = re.sub(r",(\s*[}\]])", r"\1", s)                 # drop trailing commas before } or ]
    try:
        return json.loads(s2)
    except json.JSONDecodeError:
        pass
    try:                                                  # last resort: repair common LLM-JSON slips
        from json_repair import repair_json               # (unescaped quotes, missing commas, ...)
        return json.loads(repair_json(s))
    except Exception as e:
        fr = None
        try: fr = resp.candidates[0].finish_reason
        except Exception: pass
        try:                                              # dump the raw text so you can inspect the bad span
            _dbg = VOCAB_PATH.parent / "last_bad_llm_json.txt"; _dbg.write_text(s); _hint = f" raw saved -> {_dbg}."
        except Exception:
            _hint = ""
        truncated = "MAX_TOKENS" in str(fr)
        why = ("Output was TRUNCATED (MAX_TOKENS) — split the article." if truncated
               else "Output finished (STOP) but the JSON is malformed. Run the %pip cell to install json-repair "
                    "so the repair fallback can fix it, else inspect the saved raw JSON.")
        raise ValueError(f"Gemini JSON unparseable (finish_reason={fr}; {len(s)} chars).{_hint} {why}"
                         f"\njson error: {e}\n...tail:\n{s[-300:]}") from None


# --- select the backend ---------------------------------------------------------------------
# Must stay LAST in this cell: the Gemini `gemini_json` above is redefined every time the cell
# runs, so whichever binding happens last wins. In a separate cell this would be silently undone.
# activate() sets gemini_json, GEMINI_MODEL, RESULTS_DIR, MAX_OUTPUT_TOKENS and THINKING_BUDGET.
import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend
importlib.reload(nb_parallel); importlib.reload(llm_backend)   # plain `import` is a no-op once cached
nb_parallel.install()           # thread-aware stdout + the token/latency ledger used by usage_report

llm_backend.activate(BACKEND, ROOT, globals(), results=RESULTS_MODEL)


backend: GEMINI gemini-2.5-flash  ->  results/gemini-2.5-flash/


{'backend': 'gemini-2.5-flash',
 'kind': 'gemini',
 'model': 'gemini-2.5-flash',
 'results_dir': PosixPath('/Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/gemini-2.5-flash')}

## Inspect retrieval — top-k candidates by embedding similarity (no LLM)

Preview **Hook A on its own**: for one article, show the top-`TOP_K_MATCH` candidate properties each
relation phrase retrieves from **schema.org ∪ current property-vocabulary**, ranked by cosine
similarity — *before* the LLM judges or mints anything. On a fresh run the vocabulary is empty, so the
candidates are schema.org-only; re-run this cell after **Build** (below) to see minted `ex:` properties
surface as candidates too. No Gemini call — pure retrieval.

In [22]:
# Retrieval-only preview: the ranked candidates per relation phrase, with NO Gemini call. Uses the
# CURRENT vocab state (load_vocab(); empty => schema.org-only). This is exactly the `cands` that would
# be handed to the LLM judge in process_article — inspect it first to sanity-check what the embedder
# surfaces. Set INSPECT_ARTICLE to any id in ARTICLE_ORDER.
INSPECT_ARTICLE = ARTICLE_ORDER[0]

def preview_retrieval(aid, vocab=None, top_k=TOP_K_MATCH, show_comment=True):
    vocab = load_vocab() if vocab is None else vocab
    oie, by_id, rinfo, phrases, qtexts, groups = load_article(aid)
    cands = retrieve_candidates(qtexts, vocab)
    print(f"{aid}: {len(phrases)} relation phrases | vocab has {len(vocab['properties'])} minted "
          f"property(ies)\n" + "=" * 92)
    for i, ph in enumerate(phrases):
        sigs = ", ".join(group_signatures(groups[ph], by_id))
        print(f'\n"{ph}"   ({len(groups[ph])} instance(s) | observed {sigs})')
        print(f'   query: {qtexts[i]}')
        for k, c in enumerate(cands[i][:top_k], 1):
            inv = f'   inverse: {c["inverse_of"]}' if c.get("inverse_of") else ""
            via = f'\n          via anchor: {c["matched_anchor"]!r}' if c.get("matched_anchor") else ""
            print(f'   {k}. [{c["source"]:9}] {c["label"]:<26} score={c["score"]:.3f}   sig: {c.get("signature","")}{inv}{via}')
            if show_comment and c.get("comment"):
                print(f'          {c["comment"][:100]}')
    return cands

_ = preview_retrieval(INSPECT_ARTICLE)

  anchor pool [original]: 1521 vectors for 1521 properties  (1.0 per property)
loading reranker cross-encoder/ms-marco-MiniLM-L-6-v2 ...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

art1: 47 relation phrases | vocab has 0 minted property(ies)

"about"   (21 instance(s) | observed Statement → ExcessDeathsEvent, ExcessDeathsEvent → Country, Warning → ExtremeWeatherForecastEvent, Heatwave → Continent, ExtremeWeatherForecastEvent → Continent, DrowningDeathsEvent → Country, WildfireEvent → Country, EvacuationEvent → NaturalDisaster, ForecastStatement → ExtremeWeatherForecastEvent, Report → Continent, Announcement → ExcessDeathsEvent, Statement → DrowningDeathsEvent, Statement → WildfireEvent, Declaration → ExtremeWeatherForecastEvent)
   query: about: Indicates the primary subject or topic of an event or claim.
   1. [schema.org] mentions                   score=2.269   sig: CreativeWork → Thing
          A mention relation between a creative work and a thing it references without necessarily being about
   2. [schema.org] mainEntity                 score=1.756   sig: CreativeWork → Thing   inverse: mainEntityOfPage
          A main entity relation between a creative w

## Retrieval variants — `original` vs `A` vs `B`, side by side (no LLM calls)

Writes one JSON per mode with the full top-10 per phrase, plus a single CSV with the three
modes across the columns — open it in Excel and read each phrase's three candidate lists
against each other. `*_via` names the anchor that won; `*_new` flags a candidate the
current pool never surfaced.

In [23]:
# # --- Compare the retrieval variants: original vs A vs B, NO LLM calls --------------------------
# # Same phrases, same top-k, and the SAME vocabulary state for all three (retrieval depends on the
# # vocab, which grows during a build — comparing modes against different vocab states would measure
# # the wrong thing). Writes one JSON per mode with the full top-10 records, plus a single side-by-side
# # CSV to open in Excel: one row per (article, phrase, rank), the three modes across the columns.
# import csv

# COMPARE_MODES  = ["original", "A", "B", "C"]
# COMPARE_RERANK = ["original"]       # base modes to ALSO run through the cross-encoder ("rerank:X")
# COMPARE_TOP_K  = 10
# ALL_SPECS = COMPARE_MODES + [f"rerank:{m}" for m in COMPARE_RERANK]
# CMP_DIR = RESULTS_DIR / "_global" / "e2e"
# CMP_DIR.mkdir(parents=True, exist_ok=True)

# # Which document pool to compare over. "schema" isolates the question this experiment actually
# # asks — do the anchors surface the right SCHEMA.ORG property — with no grown vocabulary competing
# # for the ten slots (6j showed the vocab crowds schema.org out of a merged top-10). "current" adds
# # the saved vocabulary, reproducing the pool the selection LLM really sees mid-build.
# # Either way all three modes share ONE pool, or the comparison measures the wrong thing.
# COMPARE_VOCAB = "schema"           # "schema" -> schema.org only | "current" -> + saved vocabulary

# _cmp_vocab = {"properties": []} if COMPARE_VOCAB == "schema" else load_vocab()
# _pool_sfx  = "_schemaonly" if COMPARE_VOCAB == "schema" else ""   # never clobber the other run
# print(f"comparing {COMPARE_MODES} over {len(ARTICLE_ORDER)} article(s) | pool: "
#       f"schema.org ({len(schema_pool)} properties)"
#       + (f" + {len(_cmp_vocab['properties'])} minted vocab" if _cmp_vocab["properties"] else "")
#       + "\n")

# per_mode = {m: {} for m in ALL_SPECS}
# rows = []
# for aid in ARTICLE_ORDER:
#     oie, by_id, rinfo, phrases, qtexts, groups = load_article(aid)
#     cands = {m: candidates_for(m, qtexts, _cmp_vocab, COMPARE_TOP_K, phrases=phrases)
#              for m in ALL_SPECS}
#     for m in ALL_SPECS:
#         per_mode[m][aid] = [{"relation_phrase": ph, "query": qtexts[i],
#                              "observed_signatures": group_signatures(groups[ph], by_id),
#                              "n_instances": len(groups[ph]), "candidates": cands[m][i]}
#                             for i, ph in enumerate(phrases)]
#     for i, ph in enumerate(phrases):
#         seen_in_original = {c["label"] for c in cands["original"][i]}
#         for r in range(COMPARE_TOP_K):
#             row = {"article": aid, "relation_phrase": ph, "query": qtexts[i], "rank": r + 1}
#             for m in ALL_SPECS:
#                 c = cands[m][i][r] if r < len(cands[m][i]) else None
#                 k = _col(m)
#                 row[f"{k}_label"]  = c["label"] if c else ""
#                 row[f"{k}_source"] = c["source"] if c else ""
#                 row[f"{k}_score"]  = c["score"] if c else ""
#                 # which anchor won, and whether this candidate is one the current pool never showed
#                 row[f"{k}_via"]    = (c.get("matched_anchor") or "") if c else ""
#                 row[f"{k}_new"]    = ("" if (m == "original" or not c)
#                                       else ("NEW" if c["label"] not in seen_in_original else ""))
#                 if c and "embed_rank" in c:      # reranked: where e5 had put it before
#                     row[f"{k}_was"] = c["embed_rank"]
#             rows.append(row)
#     print(f"  {aid:6} {len(phrases):3} phrases")

# # --- per-mode JSON (full records: score, signature, parent, comment, winning anchor) -----------
# for m in ALL_SPECS:
#     p = CMP_DIR / f"retrieval_candidates{_sfx}{_pool_sfx}_{_col(m)}.json"
#     p.write_text(json.dumps({"mode": m, "top_k": COMPARE_TOP_K, "match_model": MATCH_MODEL,
#                              "rerank_model": RERANK_MODEL if m.startswith("rerank:") else None,
#                              "rerank_from": RERANK_FROM if m.startswith("rerank:") else None,
#                              "anchor_penalty": ANCHOR_PENALTY,
#                              "drop_ambiguous_synonyms": DROP_AMBIGUOUS_SYNONYMS,
#                              "compare_vocab": COMPARE_VOCAB,
#                              "vocab_size": len(_cmp_vocab["properties"]),
#                              "articles": per_mode[m]}, indent=2, ensure_ascii=False))
#     print("wrote", p.relative_to(ROOT))

# # --- one side-by-side CSV ----------------------------------------------------------------------
# cols = ["article", "relation_phrase", "query", "rank"] + [
#     f"{_col(m)}_{f}" for m in ALL_SPECS
#     for f in (("label", "source", "score", "via", "new", "was") if m.startswith("rerank:")
#               else ("label", "source", "score", "via", "new"))]
# csv_path = CMP_DIR / f"retrieval_candidates{_sfx}{_pool_sfx}_comparison.csv"
# with csv_path.open("w", newline="", encoding="utf-8-sig") as f:      # utf-8-sig so Excel reads it
#     w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
#     w.writeheader()
#     w.writerows(rows)
# print("wrote", csv_path.relative_to(ROOT), f"({len(rows)} rows)")

# # --- quick read: did anything actually move? ---------------------------------------------------
# top1 = {m: [] for m in ALL_SPECS}
# schema_share, newly = {m: 0 for m in ALL_SPECS}, {m: 0 for m in ALL_SPECS}
# n_phrases = 0
# for aid in ARTICLE_ORDER:
#     for i in range(len(per_mode["original"][aid])):
#         n_phrases += 1
#         for m in ALL_SPECS:
#             cs = per_mode[m][aid][i]["candidates"]
#             top1[m].append(cs[0]["label"] if cs else None)
#             schema_share[m] += sum(1 for c in cs if c["source"] == "schema.org")
#             if m != "original":
#                 orig = {c["label"] for c in per_mode["original"][aid][i]["candidates"]}
#                 newly[m] += sum(1 for c in cs if c["label"] not in orig)

# hdr = f'{"mode":18}{"top-1 changed":>15}{"schema.org in top-10":>22}{"new candidates surfaced":>25}'
# print("\n" + hdr); print("-" * len(hdr))
# for m in ALL_SPECS:
#     chg = "-" if m == "original" else sum(1 for a, b in zip(top1["original"], top1[m]) if a != b)
#     print(f'{m:18}{str(chg):>15}{f"{schema_share[m]}/{n_phrases * COMPARE_TOP_K}":>22}'
#           f'{("-" if m == "original" else newly[m]):>25}')
# print(f"\n{n_phrases} phrases compared. A high 'top-1 changed' means the anchors reordered the pool; "
#       f"whether that is BETTER is decided by 6f-6k's mint-rate table, not by this cell.")


In [24]:
# ==== EDIT ME — prompt ===================================================================
# PROPERTY_SELECT_PROMPT is the property-side twin of nb4's TYPE selection task: canonicalize each
# free-form relation phrase to a schema.org OR vocabulary property, else mint a new ex: property.
# (Relation-instance dedup is deterministic — collapsing identical subject/object pairs onto one
# triple — so it is done in code, not by the LLM; this prompt only chooses the property.)
PROPERTY_SELECT_PROMPT = """You are an expert knowledge-engineering assistant. You are given several groups of relation instances from one document. Each group shares a single free-form relation phrase (an OIE relation label). For each group independently, assign every instance a final property — do NOT merge or compare instances across different groups.

INPUT — each group provides:
- RELATION_PHRASE: the free-form relation label these instances were given, plus its definition. A relation reads subject --RELATION_PHRASE--> object.
- OBSERVED_SIGNATURES: the subject-type -> object-type signatures actually seen among the instances (helps you check domain/range fit and direction).
- CANDIDATES: a ranked list of candidate properties proposed as matches. Each candidate is either an existing Schema.org property or an existing property from our extended vocabulary; its source is marked [schema.org] or [vocab]. Each shows its `signature` (domain -> range, i.e. the subject types it applies to and the object types it accepts) and its `parent` (its super-property via subPropertyOf, if any).
- INSTANCES: a NUMBERED list of the actual subject --> object pairs (with their entity types) that were assigned this relation phrase.

ASSIGNMENTS — the unit you output:
An assignment is a subset of the group's instances (referenced by their instance NUMBERS) plus the single property chosen for that subset. Every instance MUST belong to EXACTLY ONE assignment. By default put the whole group in ONE assignment (one property for all its instances). Split into separate assignments ONLY when different instances genuinely fit different properties — typically because their subject/object types differ (see OBSERVED_SIGNATURES), so a different candidate fits each subset better. Do NOT split arbitrarily: instances that fit the same property stay together.

DECISION PROCEDURE — for each assignment, choose its property by trying these in order (reusing an existing property is strongly preferred over minting):
1. Reuse a candidate AS_IS (direct): pick the candidate whose meaning (from its comment) best fits the relation phrase and the instances and that already runs subject --> object with our arguments; set "direction" to "as_is". What matters most is that the candidate captures the SAME relation meaning. When a Schema.org candidate and a vocabulary candidate both fit equally well, prefer the schema.org one (schema.org-first).
2. Reuse a candidate INVERSE (swap): if no candidate runs subject --> object but one expresses the inverse relation, object --> subject (its domain broadly aligns with our object type and its range with our subject type), reuse it rather than mint a redundant new property — select it and set "direction" to "inverse". Choosing "inverse" means we will swap subject and object so the stored triple matches the property's direction. (A candidate may also list `inverse: <name>` — the schema.org property for the opposite direction; if that inverse property is itself the best fit and appears in the list, prefer selecting it directly with "as_is".)
3. MINT a new property: only when neither the direct nor the inverse reading of any candidate fits (see MINTING).
Judge direction from the signatures (each candidate's domain -> range vs the OBSERVED_SIGNATURES) and the instance types, not from the comment wording — candidate comments are written neutrally and may not state a direction. The stored triple MUST read subject --> object in the chosen property's own direction.

INSTANCE TEST — how to accept a candidate (in either direction): judge by the INSTANCES, not the phrase definition alone. Does the candidate express the same relation meaning for these subject/object pairs, in the orientation you chose (as_is: subject against the candidate's domain, object against its range; inverse: subject against its range, object against its domain)? Treat domain/range LOOSELY: schema.org's domain/range are permissive suggestions, and our entity types are often subtypes of, supertypes of, or siblings of a property's declared domain/range — so the signature is a SOFT compatibility signal, not a strict filter. Accept the candidate whenever it preserves the relation's meaning and the arguments are broadly compatible — do NOT reject it just because a type is more specific, more general, or a near-neighbour of the declared domain/range. Reject on domain/range grounds ONLY when the pairing is genuinely incompatible: meaningless, or a different KIND of relation that would change what the property asserts. A candidate that is somewhat broader but still preserves what makes the relation distinctive is preferable to a new property.

MINTING — mint a new property ONLY when no existing candidate captures the relation's meaning in either direction — i.e. when reusing the best candidate would change or lose the relation's meaning (including when it is only "correct" by being so broad it loses the relation's distinguishing meaning). Do NOT mint merely because the subject/object types are not an exact match for a candidate's declared domain/range: a meaning-preserving candidate with a loose type fit is preferable to a new property. When you mint:
- Name it from the instances: name the relation the instances actually exhibit (their subject-type -> object-type signatures), which need not match the relation phrase or its definition when that phrase/definition is vague or misfits the instances.
- Label format: use a concise, GENERAL, reusable camelCase property label that names the RELATION ITSELF and does NOT embed a subject or object (neither its type nor a specific entity) in the label (write "after", not "afterEvent"; "locatedIn", not "locatedInCity"; "hasValue", not "hasQuantityValue"); the object belongs in the object slot, not the predicate, and two phrasings of the same relation must yield the SAME label.
- Definition & direction: give it a one-sentence definition of the relation; a mint is always in the phrase's own direction, so its "direction" is "as_is".
- Parent, domain, range: assign a parent (the closest existing property this should be a subPropertyOf — a Schema.org or [vocab] candidate label, or any candidate's `parent`; use null if none is a sensible super-property), and domain / range (short lists of the subject types and object types the minted property applies to, taken from the OBSERVED_SIGNATURES).

OUTPUT — report each assignment as: "instances" (the list of instance NUMBERS it covers) and "chosen_property" with: source ("schema.org" or "vocab" when you chose a candidate, matching that candidate's marked source; "new" when you mint), the chosen label (the exact candidate label, or the new camelCase label), whether it is new, its "direction" ("as_is" or "inverse"), its definition / parent / domain / range (only when new; else null), and a one-line justification (mention the flip when direction is "inverse").

Return ONLY valid JSON — no prose, no code fences. Output one result object per input group, in the same order, each echoing its exact relation_phrase string. Angle-bracket placeholders show the shape, not literal values:
{"results": [{"relation_phrase": "<phrase>", "assignments": [{"instances": [1, 2], "chosen_property": {"source": "schema.org | vocab | new", "label": "<property label>", "is_new": false, "direction": "as_is", "definition": null, "parent": null, "domain": null, "range": null, "justification": "one short line"}}]}]}
Include every group exactly once, and cover every instance NUMBER of that group exactly once across its assignments."""
print("prompt loaded")

prompt loaded


In [25]:
# Hook A caller — ONE Gemini call per article. For each relation-phrase group the LLM returns one or
# more ASSIGNMENTS: each assignment is a subset of the group's instances (by 1-based number) plus the
# property chosen for that subset. This lets a single phrase split across several properties when its
# instances have different signatures. Returns: phrase -> [{"instances": [0-based idx...], "chosen": {...}}].
def llm_assign(phrases, rinfo, groups, by_id, cands):
    blocks = []
    for i, ph in enumerate(phrases):
        b = [f"GROUP {i+1} —", f'RELATION_PHRASE: "{ph}"']
        if rinfo.get(ph, {}).get("definition"):
            b.append(f'DEFINITION: {rinfo[ph]["definition"]}')
        b.append("OBSERVED_SIGNATURES: " + "; ".join(group_signatures(groups[ph], by_id)))
        b.append("CANDIDATES (ranked):")
        if cands[i]:
            for k, c in enumerate(cands[i], 1):
                par = c.get("parent") or "—"
                inv = f' | inverse: {c["inverse_of"]}' if c.get("inverse_of") else ""
                b.append(f'  {k}. [{c["source"]}] {c["label"]} | signature: {c.get("signature","")} | subPropertyOf: {par}{inv}')
                b.append(f'     {c.get("comment", "")}')
        else:
            b.append("  (none)")
        b.append("INSTANCES (numbered):")
        b += [f'  {j}. "{r["subject_text"]}" ({_etype(by_id, r["subject_id"])}) --> "{r["object_text"]}" ({_etype(by_id, r["object_id"])})'
              for j, r in enumerate(groups[ph], 1)]
        blocks.append("\n".join(b))
    user = f"You are given {len(phrases)} GROUPS from one document.\n\n" + "\n\n".join(blocks)
    parsed  = parse_json(gemini_json(PROPERTY_SELECT_PROMPT, user))
    results = parsed.get("results", parsed) if isinstance(parsed, dict) else parsed
    raw = {r.get("relation_phrase"): r for r in (results or []) if r.get("relation_phrase") in set(phrases)}

    out = {}
    for i, ph in enumerate(phrases):
        n   = len(groups[ph])
        top = cands[i][0] if cands[i] else {"source": "schema.org", "label": "relatedTo"}
        fallback = {"source": top["source"], "label": top["label"], "is_new": False,
                    "direction": "as_is", "justification": "fallback: top candidate"}
        asgs, seen = [], set()
        for a in ((raw.get(ph) or {}).get("assignments") or []):
            ch, idxs = (a.get("chosen_property") or {}), []
            for x in (a.get("instances") or []):
                try:
                    j = int(x) - 1
                except (TypeError, ValueError):
                    continue
                if 0 <= j < n and j not in seen:          # valid, first assignment wins on any overlap
                    seen.add(j); idxs.append(j)
            if idxs:
                asgs.append({"instances": idxs, "chosen": ch})
        missing = [j for j in range(n) if j not in seen]  # any uncovered instance -> fallback assignment
        if missing:
            asgs.append({"instances": missing, "chosen": fallback})
        out[ph] = asgs or [{"instances": list(range(n)), "chosen": fallback}]
    return out

# ---- Minting: admit a chosen "new" property into the vocabulary -------------------------------
def _schema_ref(label):
    return {"label": label, "source": "schema.org", "vocab_id": None,
            "definition": None, "parent": None, "domain": None, "range": None, "is_new_to_vocab": False}

def _vocab_ref(ent, is_new=False):
    return {"label": ent["canonical_label"], "source": "vocab", "vocab_id": ent["id"],
            "definition": ent["definition"], "parent": ent["parent"],
            "domain": ent.get("domain"), "range": ent.get("range"), "is_new_to_vocab": is_new}

# Admit a minted property as a NEW vocabulary entry (or, in restrict mode, fall back to its parent /
# a generic related-to link). Parent (subPropertyOf) may be null — properties have no single root.
def _apply_mint(aid, phrase, ch, group, vocab):
    label = ch.get("label"); definition = ch.get("definition") or ""
    parent = ch.get("parent")
    if parent is not None and not (parent in SCHEMA_LABELS or _is_vocab_label(vocab, parent)):
        parent = None                                          # unknown parent -> no super-property
    if not EXPAND_MODE:                                        # restrict mode: fall back to the parent
        return _schema_ref(parent) if parent in SCHEMA_LABELS else _schema_ref("relatedTo")
    ent = {"id": f"P{vocab['next_id']:04d}", "canonical_label": label, "definition": definition,
           "parent": parent, "domain": ch.get("domain") or [], "range": ch.get("range") or [],
           "aliases": [], "support": len(group), "first_seen": aid,
           "provenance": [{"article": aid, "relation_phrase": phrase,
                           "instances": [f'{i["subject_text"]} -> {i["object_text"]}' for i in group]}]}
    vocab["properties"].append(ent); vocab["next_id"] += 1
    return _vocab_ref(ent, is_new=True)

In [26]:
# Build canonical relation records from per-instance assignments. `inst_assignments` is a flat list of
# (phrase, instance, tref) — each instance carries the property (tref) chosen for ITS assignment, so two
# instances of the same phrase can land on different properties. Each instance is ORIENTED to its
# property's direction (subject/object SWAPPED when tref direction == "inverse"), then instances are
# DEDUPED deterministically on (subject_id, property, object_id): identical triples (e.g. a direct and an
# inverse phrase that resolve to the same property between the same two entities) collapse into ONE
# record, evidence unioned. No LLM here.
def _build_relation_records(by_id, inst_assignments):
    records = {}   # (subject_id, property_label, object_id) -> record
    for ph, r, tref in inst_assignments:
        flip = tref.get("direction") == "inverse"
        if flip:                                                  # swap to the property's direction
            s_id, s_txt, o_id, o_txt = r["object_id"], r["object_text"], r["subject_id"], r["subject_text"]
        else:
            s_id, s_txt, o_id, o_txt = r["subject_id"], r["subject_text"], r["object_id"], r["object_text"]
        key = (s_id, tref.get("label"), o_id)
        rec = records.get(key)
        if rec is None:
            rec = {
                "subject_id":   s_id,
                "subject_text": s_txt,
                "subject_type": _etype(by_id, s_id),
                "property":                 tref.get("label"),
                "property_source":          tref.get("source"),
                "vocab_id":                 tref.get("vocab_id"),
                "property_is_new_to_vocab": tref.get("is_new_to_vocab", False),
                "property_definition":      tref.get("definition"),
                "property_parent":          tref.get("parent"),
                "property_justification":   tref.get("justification"),
                "object_id":    o_id,
                "object_text":  o_txt,
                "object_type":  _etype(by_id, o_id),
                "relation_phrases": [],          # the free-form phrase(s) that produced this triple
                "phrase_directions": {},         # phrase -> "as_is" | "inverse" (was it flipped?)
                "evidence":         [],
                "confidence":       0.0,
            }
            records[key] = rec
        if ph not in rec["relation_phrases"]:
            rec["relation_phrases"].append(ph)
        rec["phrase_directions"][ph] = "inverse" if flip else "as_is"
        ev = r.get("evidence")
        if ev and ev not in rec["evidence"]:
            rec["evidence"].append(ev)
        rec["confidence"] = max(rec["confidence"], float(r.get("confidence") or 0.0))
    return list(records.values())

In [27]:
# Process ONE article end-to-end, self-contained:
#   retrieve (Hook A) -> ONE LLM call assigning each instance a property (a phrase may split across
#   several properties) -> admit any minted properties as new vocabulary entries -> build canonical
#   (deduped) relation records. Writes nary_graph_ext.json AND nary_relation_matching_review.json (per
#   relation phrase: query text, the candidates offered, and EACH assignment's instance subset + chosen property).
def process_article(aid, vocab, write=True, save=True, verbose=True):
    oie, by_id, rinfo, phrases, qtexts, groups = load_article(aid)
    cands  = retrieve_candidates(qtexts, vocab)
    report = llm_assign(phrases, rinfo, groups, by_id, cands)   # phrase -> [assignments]

    assigns = {}                                 # phrase -> [{"instances": [r...], "tref": tref, "chosen": ch}]
    inst_res, n_mint, n_flip = [], 0, 0
    for ph in phrases:
        rows = []
        for a in report[ph]:
            ch    = a["chosen"]
            insts = [groups[ph][j] for j in a["instances"]]
            is_mint  = ch.get("is_new") or ch.get("source") == "new"
            existing = _find_vocab_by_label(vocab, ch.get("label"))   # reuse an existing entry if the label matches
            # a mint is in the phrase's own direction; a reused candidate may be flipped to inverse
            direction = "as_is" if (is_mint and not existing) else ("inverse" if ch.get("direction") == "inverse" else "as_is")
            if is_mint and not existing:                              # admit a genuinely new property
                before = len(vocab["properties"])
                tref = _apply_mint(aid, ph, ch, insts, vocab)
                n_mint += (len(vocab["properties"]) > before)
            elif existing:                                            # chose (or minted a dup of) an existing vocab property
                _bump(existing, aid, ph, insts, flip=(direction == "inverse")); tref = _vocab_ref(existing)
            else:                                                     # a schema.org candidate
                tref = _schema_ref(ch.get("label"))
            tref["direction"] = direction
            tref["justification"] = ch.get("justification")
            n_flip += (direction == "inverse") * len(insts)
            for r in insts:
                inst_res.append((ph, r, tref))
            rows.append({"instances": insts, "tref": tref, "chosen": ch})
        assigns[ph] = rows

    records = _build_relation_records(by_id, inst_res)

    if write:
        out_dir = RESULTS_DIR / aid
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / f"e2e/nary_graph{_osfx}_extended.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE,
             "entities": oie.get("entities", []), "nary_nodes": oie.get("nary_nodes", []),
             "relations": records},
            indent=2, ensure_ascii=False))
        # Review file: one row per relation phrase, listing EACH assignment (its instance subset + the
        # property chosen for it, with super-property path) so a phrase split across properties is fully
        # auditable, plus the candidates the LLM chose from at THIS article's vocab state.
        def _chosen_block(tref):
            return {"label": tref.get("label"), "source": tref.get("source"), "direction": tref.get("direction"),
                    "vocab_id": tref.get("vocab_id"), "is_new_to_vocab": tref.get("is_new_to_vocab", False),
                    "definition": tref.get("definition"), "parent": tref.get("parent"),
                    "domain": tref.get("domain"), "range": tref.get("range"),
                    "path": _full_path(tref, vocab), "justification": tref.get("justification")}
        review = [{
            "relation_phrase":      ph,
            "definition":           rinfo.get(ph, {}).get("definition"),
            "embedding_query_text": qtexts[i],
            "observed_signatures":  group_signatures(groups[ph], by_id),
            "n_assignments":        len(assigns[ph]),
            "assignments": [{
                "instances":       [f'{r["subject_text"]} -> {r["object_text"]}' for r in a["instances"]],
                "chosen_property": _chosen_block(a["tref"]),
                "llm_proposed":    {k: (a["chosen"] or {}).get(k) for k in ("source", "label", "is_new", "direction")},
            } for a in assigns[ph]],
            "candidates":           cands[i],
        } for i, ph in enumerate(phrases)]
        (out_dir / f"e2e/nary_relation_matching_review{_osfx}.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE, "relations": review},
            indent=2, ensure_ascii=False))
    if save:
        save_vocab(vocab)
    if verbose:
        merged  = sum(1 for r in records if len(r["relation_phrases"]) > 1 or len(r["evidence"]) > 1)
        n_split = sum(1 for ph in phrases if len(assigns[ph]) > 1)
        n_inst  = len(oie["relations"])
        print(f"  {aid}: {len(phrases)} phrases ({n_split} split), {n_inst} instances -> {len(records)} triples "
              f"({merged} merged, {n_flip} inverse-reused) | +{n_mint} minted | vocab now {len(vocab['properties'])}")
    return records, vocab

## Dry run — one article, no writes

In [28]:
# # Dry run on the FIRST article against a COPY of the current vocab (no writes, no vocab mutation).
# _vocab_preview = copy.deepcopy(load_vocab())
# _records, _ = process_article(ARTICLE_ORDER[0], _vocab_preview, write=False, save=False)
# for (src, lbl), c in Counter((r.get("property_source"), r["property"]) for r in _records).most_common():
#     print(f"  {str(src):<11} {lbl}  x{c}")

## Build — incremental vocabulary over all articles (in order)

In [29]:
import time
# Incremental build. FRESH=True: start from an EMPTY vocab and process ALL of ARTICLE_ORDER (reproducible).
# FRESH=False: keep the existing relation_vocabulary.json and process only NEW_ARTICLES (future growth).
FRESH        = True
NEW_ARTICLES = []          # used only when FRESH=False

if FRESH:
    vocab = {"properties": [], "next_id": 1}; save_vocab(vocab); todo = ARTICLE_ORDER
else:
    vocab = load_vocab()
    todo  = NEW_ARTICLES or [a for a in ARTICLE_ORDER
                             if not (RESULTS_DIR / a / f"e2e/nary_graph{_osfx}_extended.json").exists()]

print(f"building property vocabulary over {len(todo)} article(s): {todo}\n")
import time
nb_parallel.reset_usage()          # the report below covers only the calls THIS run makes
_t0 = time.perf_counter()

LAST_USAGE = None
TOKENS = []
for aid in todo:
    _ta = time.perf_counter()
    with nb_parallel.usage_scope(aid):      # tag this article's calls so the report breaks down per article
        process_article(aid, vocab, write=True, save=True, verbose=True)
    TOKENS.append({"article": aid, "seconds": round(time.perf_counter() - _ta, 1),
                   **(LAST_USAGE or {})})   # tokens (incl. THINKING) + elapsed for that article
    time.sleep(5)
print(f"\ndone — {len(vocab['properties'])} canonical properties in {VOCAB_PATH}")
# token-usage log (read by 6f's comparison cell) — SEPARATE per method/variant
_toktot = {k: sum((t.get(k) or 0) for t in TOKENS)
           for k in ("prompt_tokens", "output_tokens", "thoughts_tokens", "total_tokens", "seconds")}
_tokp = RESULTS_DIR / "_global" / "e2e" / f"token_usage{_osfx}.json"
_tokp.write_text(json.dumps({"method": "candidate", "variant": VARIANT, "model": GEMINI_MODEL,
                             "per_article": TOKENS, "totals": _toktot}, indent=2, ensure_ascii=False))
print(f"tokens (candidate): {_toktot} -> {_tokp.name}")

# --- what this run cost: calls, prompt / output / THINKING tokens, and elapsed time ------------
nb_parallel.usage_report("6e · e2e relation canonicalization", wall_seconds=time.perf_counter() - _t0)

building property vocabulary over 4 article(s): ['art1', 'art2', 'art4', 'art5']

  art1: 47 phrases (2 split), 156 instances -> 155 triples (1 merged, 10 inverse-reused) | +12 minted | vocab now 12
  art2: 92 phrases (5 split), 198 instances -> 198 triples (0 merged, 0 inverse-reused) | +23 minted | vocab now 35
  art4: 26 phrases (0 split), 77 instances -> 77 triples (0 merged, 0 inverse-reused) | +12 minted | vocab now 47
  art5: 48 phrases (4 split), 163 instances -> 163 triples (0 merged, 6 inverse-reused) | +34 minted | vocab now 81

done — 81 canonical properties in /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/gemini-2.5-flash/_global/e2e/nary_relation_vocabulary_dedup_rr.json
tokens (candidate): {'prompt_tokens': 230036, 'output_tokens': 29719, 'thoughts_tokens': 123640, 'total_tokens': 383395, 'seconds': 718.0} -> token_usage_dedup_rr.json

LLM USAGE — 6e · e2e relation canonicalization
               calls     prompt     output  th

{'label': '6e · e2e relation canonicalization',
 'totals': {'calls': 4,
  'prompt_tokens': 230036,
  'output_tokens': 29719,
  'thoughts_tokens': 123640,
  'total_tokens': 383395,
  'seconds': 638.3},
 'wall_seconds': 737.994565125,
 'per_tag': {'art1': {'calls': 1,
   'seconds': 155.3,
   'prompt_tokens': 58898,
   'output_tokens': 8391,
   'thoughts_tokens': 28814,
   'total_tokens': 96103},
  'art2': {'calls': 1,
   'seconds': 269.6,
   'prompt_tokens': 97957,
   'output_tokens': 7354,
   'thoughts_tokens': 58163,
   'total_tokens': 163474},
  'art4': {'calls': 1,
   'seconds': 75.6,
   'prompt_tokens': 26733,
   'output_tokens': 4189,
   'thoughts_tokens': 13355,
   'total_tokens': 44277},
  'art5': {'calls': 1,
   'seconds': 137.8,
   'prompt_tokens': 46448,
   'output_tokens': 9785,
   'thoughts_tokens': 23308,
   'total_tokens': 79541}},
 'calls': [{'prompt_tokens': 58898,
   'output_tokens': 8391,
   'thoughts_tokens': 28814,
   'total_tokens': 96103,
   'seconds': 155.34928862

## Report + verify

In [30]:
# Property vocabulary summary, then verify each ext file covers every nb6 relation instance
# (each oie relation maps to exactly one canonical triple — none lost; merges only collapse duplicates).
vocab = load_vocab()
print(f"PROPERTY VOCABULARY — {len(vocab['properties'])} canonical properties\n" + "-" * 74)
for t in sorted(vocab["properties"], key=lambda x: -x.get("support", 0)):
    al = f"  aka {t['aliases']}" if t.get("aliases") else ""
    print(f"  {t['id']}  {t['canonical_label']:<26} subPropertyOf {str(t.get('parent')):<16} "
          f"support={t.get('support')} | {_sig(t.get('domain'), t.get('range'))}{al}")

print("\nVERIFY ext vs oie_relations (every relation instance represented)\n" + "-" * 74)
ok = True
for aid in ARTICLE_ORDER:
    ext = RESULTS_DIR / aid / f"e2e/nary_graph{_osfx}_extended.json"
    if not ext.exists():
        print(f"  {aid:<8} (no ext file)"); continue
    oie = json.loads((RESULTS_DIR / aid / f"e2e/nary_graph{_osfx}_typed_consolidated.json").read_text())
    x   = json.loads(ext.read_text())["relations"]
    # every entity pair from oie must appear in some record. Compare UNORDERED pairs, since an inverse
    # reuse legitimately swaps subject/object — the pair is preserved, only its orientation changes.
    orig_pairs = {frozenset((r["subject_id"], r["object_id"])) for r in oie["relations"]}
    got_pairs  = {frozenset((r["subject_id"], r["object_id"])) for r in x}
    nvoc = sum(1 for r in x if r.get("property_source") == "vocab")
    nflip = sum(1 for r in x if "inverse" in (r.get("phrase_directions") or {}).values())
    covered = orig_pairs <= got_pairs
    ok &= covered
    print(f"  {aid:<8} {len(oie['relations']):>4} instances -> {len(x):>4} triples "
          f"({nvoc} via vocab, {nflip} inverse-reused)   {'OK' if covered else 'FAIL: pair(s) dropped'}")
print("-" * 74); print("consistent" if ok else "inconsistencies")

PROPERTY VOCABULARY — 81 canonical properties
--------------------------------------------------------------------------
  P0049  informationSource          subPropertyOf creator          support=14 | Statement/Report/Forecast/Announcement/Improvement/Warning → GovernmentOfficial/MedicalOrganization/NewsMediaOrganization/GovernmentOrganization/Consortium
  P0007  hasQuantity                subPropertyOf None             support=9 | ExcessDeathsEvent/DrowningDeathsEvent/WildfireEvent/EvacuationEvent → Quantity
  P0048  quantityLocatedIn          subPropertyOf location         support=9 | Quantity → Country
  P0053  associatedWith             subPropertyOf isRelatedTo      support=9 | Quantity → DiseaseOutbreak/DiseaseSpreadEvent
  P0042  hasPrizeForRank            subPropertyOf None             support=8 | PrizeMoneyDistributionEvent → CompetitionRank
  P0043  receivesMonetaryAmount     subPropertyOf None             support=8 | CompetitionRank → MonetaryAmount
  P0050  reportsAbout    

*(Optional) the GLOBAL re-consolidation and property-hierarchy passes from nb7 (cells after this point there) port over unchanged — they operate on `nary_relation_vocabulary.json` + the `nary_graph_ext.json` files. Copy them in when you want to collapse synonymous mints or print the property tree.*

# ─────────────────────────── GLOBAL RE-CONSOLIDATION (optional) ───────────────────────────
## Everything below is the optional global-reconsolidation block

The property-side twin of 6d's re-consolidation. Run the pipeline above (through **Build** and
**Report + verify**) to produce the raw incremental property vocabulary (`relation_vocabulary.json`).
The cells below are a **self-contained, optional block** — run them to clean the vocabulary globally
(merge synonymous properties, or link a specific property under a broader one as `subPropertyOf`), or
**skip them entirely** to use the raw vocabulary as-is. This block writes only **NEW files** — every
incremental output is left untouched, so you can always compare before/after.

**Candidate pool = vocabulary ∪ schema.org (mixed).** Every vocabulary property is embedded and
shortlisted against *one ranked pool* holding both the other minted properties and the full schema.org
property pool (`CONSOLIDATE_TOP_K` nearest neighbours each, `CONSOLIDATE_INCLUDE_SCHEMA=True`); those
neighbours are the only merge candidates the judge may use. A minted property that merely restates a
standard property can therefore be **folded into schema.org** (the extension entry disappears, its
relation instances are re-labelled to the schema.org property) or made a **`subPropertyOf` it**.
schema.org properties are fixed — survivor / parent only, never merged away, so only `P…` ids appear in
`merge_ids`. Direction still rules: the prompt shows each candidate's `domain → range` signature and
`inverse_of`, and forbids grouping a relation with its inverse. Set `CONSOLIDATE_INCLUDE_SCHEMA=False`
to fall back to the old vocab-only shortlist.

**Two guards on the schema.org side.** `SCHEMA_EXCLUDE_LABELS` drops schema.org's metamodel properties
(`domainIncludes`, `rangeIncludes`, `supersededBy`, `inverseOf`) from the pool — they describe the
vocabulary itself, not any relation an article states. And when a schema.org keep shares its **label**
with one of its children, that is a duplicate rather than a specialisation: the child is merged into
schema.org instead of nested under it, which would otherwise make the property its own `subPropertyOf`
(`parent` holds a bare label, and the vocab side wins the lookup) and drop it from the hierarchy. A final
sweep nulls any self-parent that slips through and lists it in the report as `self_parent_fixed`.

**Prompt layout.** One interleaved listing: each vocabulary property's line, then its candidates
indented under it as `->` lines, each carrying its own score, source, super-property, observed
`domain → range` signature, `inverse_of` (schema.org) and definition (truncated to
`CANDIDATE_DEF_CHARS`). Nothing is referenced by bare id, so the judge reads one property and its
options together — signatures sit side by side, which is exactly what the direction rule needs.

Because the LLM only sees each property's shortlist (not the full cross-product), the pass scales the
way the incremental build does — the prompt grows as `N × (1 + TOP_K)` lines.

The block, in order:
1. **Config + prompt** — `SUBPROP_MODE`, `CONSOLIDATE_TOP_K` / `CONSOLIDATE_INCLUDE_SCHEMA`, output paths, `load_consolidated`, `CONSOLIDATE_ALL_PROMPT`.
2. **`consolidate_vocab`** — mixed-pool shortlist inlined per property + ONE Gemini call → `relation_vocabulary_consolidated.json` + `relation_consolidation_report.json` (with the full `neighbour_shortlist`, each candidate tagged `vocab` / `schema.org`, and an `in_shortlist` audit flag per merge/re-parent).
3. **`refresh_ext_after_consolidation`** — no LLM; writes `nary_graph_extended_consolidated.json` per article + a changes-only `relation_consolidation_ext_changes.json`.
4. **`build_property_hierarchy`** — indented schema.org + extension property tree, into `property_hierarchy_ext.txt` (raw) and `property_hierarchy_ext_consolidated.txt` (consolidated).

In [31]:
# Global consolidation prompt — used by the ONE-call consolidate_vocab below.
# SUBPROP_MODE controls how a SPECIFIC property is related to a broader property ALREADY in the pool:
#   "off"        : only merge exact synonyms (no broader/narrower handling).
#   "subproperty": KEEP the specific property, set its parent (subPropertyOf) to the broader
#                  property — builds an is-a-kind-of-relation hierarchy; preserves granularity.
#   "merge"      : COLLAPSE the specific property into the broader parent (child removed, folded in).
SUBPROP_MODE = "subproperty"
assert SUBPROP_MODE in ("off", "subproperty", "merge"), "SUBPROP_MODE must be 'off', 'subproperty', or 'merge'"
CONSOLIDATE_TOP_K = 5   # retrieve-then-judge: nearest-neighbour shortlist size per item
# Shortlist each vocab property against the FULL MIXED POOL (other vocab properties u the whole
# schema.org property pool) instead of vocab-only. A minted property that duplicates a schema.org
# property can then be folded INTO schema.org (or parented under it), not just merged with its
# siblings. schema.org properties are fixed: they can only ever be the SURVIVOR / PARENT of a group.
CONSOLIDATE_INCLUDE_SCHEMA = True
SCHEMA_ID_PREFIX = "schema:"    # how a schema.org candidate is identified to the LLM (e.g. schema:author)
CANDIDATE_DEF_CHARS = 200       # per-candidate definition budget (candidates are inlined under each property)
# schema.org METAMODEL properties describe the vocabulary itself (which class a property belongs to,
# what supersedes what) rather than any relation an article states, so they are dropped from the
# candidate pool — the class-side twin of 6d's SCHEMA_EXCLUDE_LABELS. Empty the set to allow them back.
SCHEMA_EXCLUDE_LABELS = {"domainIncludes", "rangeIncludes", "supersededBy", "inverseOf"}

# Consolidation writes to SEPARATE files — the raw relation_vocabulary.json is left INTACT.
CONSOLIDATED_VOCAB_PATH   = RESULTS_DIR / f"_global/e2e/nary_relation_vocabulary{_osfx}_consolidated.json"   # consolidated snapshot
CONSOLIDATION_REPORT_PATH = RESULTS_DIR / f"_global/e2e/nary_relation_consolidation_report{_osfx}.json"      # what merged / re-parented
def load_consolidated():
    """The consolidated vocab if it exists, else fall back to the raw incremental one."""
    return json.loads(CONSOLIDATED_VOCAB_PATH.read_text()) if CONSOLIDATED_VOCAB_PATH.exists() else load_vocab()

CONSOLIDATE_ALL_PROMPT = """You are organizing a vocabulary of relation properties. You are given every vocabulary property (id, label, parent, support, domain -> range signature, definition), and directly under each one its CANDIDATES: the most similar entries retrieved for it, each with its own signature and definition. Each property expresses a directed relation subject --property--> object. Produce groups of two kinds:

SYNONYM — two or more entries that denote the same relation (true synonyms: the same relation in the same direction, differing only in wording or label). Set keep_id to the entry that should survive and merge_ids to the rest.
SUBPROPERTY — one or more specific (child) vocabulary properties that are all a more specific kind of a single more general (parent) relation — an is-a relationship between relations, in the same direction. Set keep_id to the general parent property's id and merge_ids to the specific child ones. Only relate a child to a parent that genuinely subsumes it in the same direction — every instance under a child MUST still be a valid instance of the parent relation.

A property may only be grouped with entries listed among ITS OWN candidates, or with a property that lists IT among that property's candidates. Never propose a pair that appears in neither list, and never invent an id.

TWO ID NAMESPACES:
- `P...` ids are VOCABULARY properties — extension properties minted by this pipeline. Only these may appear in merge_ids.
- `schema:x` ids are SCHEMA.ORG properties — a fixed, standard vocabulary. They are NOT part of the extension and can NEVER be merged away or removed, so a `schema:x` id may appear ONLY as keep_id, never in merge_ids.

Prefer schema.org whenever it fits: it is the standard vocabulary, and an extension property that merely restates a schema.org property is redundant.
- SYNONYM with keep_id = `schema:x` means the vocabulary propertie(s) in merge_ids are redundant restatements of that standard property; every relation instance using them will be re-labelled to schema.org x. Only do this when the schema.org property denotes the same relation IN THE SAME DIRECTION (subject and object stay where they are) — never when it is the inverse, and never when it is merely broader.
- SUBPROPERTY with keep_id = `schema:x` means the vocabulary child properties are more specific kinds of that standard property, in the same direction, and will be attached under it as subPropertyOf.
- If a vocabulary entry carries the SAME label as a schema.org candidate, that is a duplicate of the standard property, not a specialisation of it: group them as SYNONYM with the `schema:x` id as keep_id.

Rules:
- Judge by each entry's label, definition, and domain/range direction (not the label alone). Be conservative. For SYNONYM, only group true equivalents in the same direction (never group a relation with its inverse — the `inverse_of` field flags schema.org properties that have one). For SUBPROPERTY, only relate a child to a parent when the parent clearly and safely subsumes it in the same direction; if a real distinction would be lost, leave them separate.
- Each vocabulary property belongs to at most one group. Every group has a keep_id and one or more merge_ids. Give a one-line reason and set "kind" to "synonym" or "subproperty".

Return ONLY valid JSON — no prose, no code fences. Angle-bracket placeholders show the shape, not literal values:
{"groups": [{"kind": "subproperty", "keep_id": "<parent id>", "merge_ids": ["<child id>"], "reason": "one short line"}, {"kind": "synonym", "keep_id": "<schema:property or id>", "merge_ids": ["<id>"], "reason": "one short line"}]}"""
print(f"consolidation prompt loaded | SUBPROP_MODE={SUBPROP_MODE!r} | "
      f"mixed pool (schema.org candidates): {CONSOLIDATE_INCLUDE_SCHEMA} | consolidated -> {CONSOLIDATED_VOCAB_PATH.name}")

consolidation prompt loaded | SUBPROP_MODE='subproperty' | mixed pool (schema.org candidates): True | consolidated -> nary_relation_vocabulary_dedup_rr_consolidated.json


In [32]:
import time

# Retrieve-then-judge over the MIXED pool: every vocab property is shortlisted against the other vocab
# properties AND the full schema.org property pool in ONE ranking, so the judge sees standard
# properties and sibling extension properties side by side as merge candidates.
def _candidate_line(src, obj, score):
    """One rendered candidate, inlined under the property it was retrieved for — carries everything the
    judge needs (id, label, super-property, domain -> range signature, inverse, definition), so no
    separate lookup block is needed."""
    if src == "schema.org":
        desc = (obj.get("enriched_comment") or obj.get("comment") or "").replace("\n", " ").strip()
        inv  = f' | inverse_of: {obj.get("inverse_of")}' if obj.get("inverse_of") else ""
        return (f'    -> {SCHEMA_ID_PREFIX}{obj["label"]} ({score}) | schema.org'
                f' | parent: {_schema_super(obj["label"])} | {obj.get("signature", "")}{inv}'
                f' | def: {desc[:CANDIDATE_DEF_CHARS]}')
    desc = (obj.get("definition") or "").replace("\n", " ").strip()
    return (f'    -> {obj["id"]} ({score}) | vocab | label: {obj["canonical_label"]}'
            f' | parent: {obj.get("parent")} | support: {obj.get("support", 0)}'
            f' | {_sig(obj.get("domain"), obj.get("range"))}'
            f' | def: {desc[:CANDIDATE_DEF_CHARS]}')

def _consolidation_shortlist(ents_sorted):
    """Top-CONSOLIDATE_TOP_K nearest neighbours of each vocab property over (vocab \\ self) u schema.org.
    Returns (cand_lines, shortlist_records, adj): `cand_lines` maps a vocab id to its rendered candidate
    lines (spliced under that property's line in the prompt), `adj` maps it to the set of candidate ids
    the embedder proposed (used for the audit), `shortlist_records` is the same thing for the report."""
    ids  = [t["id"] for t in ents_sorted]
    labs = [t["canonical_label"] for t in ents_sorted]
    V    = embed([vocab_embedding_text(t) for t in ents_sorted], DP)
    meta = [("vocab", ids[i], labs[i], ents_sorted[i]) for i in range(len(ents_sorted))]
    D    = V
    if CONSOLIDATE_INCLUDE_SCHEMA:
        keep = [i for i, p in enumerate(schema_pool) if p["label"] not in SCHEMA_EXCLUDE_LABELS]
        D    = np.vstack([V, schema_emb[keep]])
        meta = meta + [("schema.org", SCHEMA_ID_PREFIX + schema_pool[i]["label"],
                        schema_pool[i]["label"], schema_pool[i]) for i in keep]
    sims = V @ D.T
    for i in range(len(ents_sorted)):
        sims[i, i] = -1.0                                   # a property is never its own candidate
    k = min(CONSOLIDATE_TOP_K, D.shape[0] - 1)

    cand_lines, shortlist, adj = {}, [], {}
    for i in range(len(ents_sorted)):
        top   = np.argsort(-sims[i])[:k]
        cands = [(meta[j], round(float(sims[i][j]), 3)) for j in top]
        adj[ids[i]] = {m[1] for m, _ in cands}
        cand_lines[ids[i]] = [_candidate_line(m[0], m[3], s) for m, s in cands]
        shortlist.append({"id": ids[i], "label": labs[i],
                          "neighbours": [{"id": m[1], "label": m[2], "source": m[0], "score": s}
                                         for m, s in cands]})
    return cand_lines, shortlist, adj

def consolidate_vocab(vocab, write_report=True, verbose=True):
    """Global re-consolidation of the PROPERTY vocabulary in ONE Gemini call, judged against the MIXED
    candidate pool (vocab u schema.org). The model returns SYNONYM and SUBPROPERTY groups whose keep_id
    may be a vocab id OR a `schema:x` property; how SUBPROPERTY (is-a) groups are applied depends on
    SUBPROP_MODE:
      "off"         -> ignore them (only synonyms merge).
      "subproperty" -> KEEP each child, set its parent (subPropertyOf) to the broader property
                       (vocab OR schema.org).
      "merge"       -> fold each child into the broader parent (removed; label -> alias).
    SYNONYM groups merge onto the highest-support member, or onto the schema.org property when the LLM
    kept one (the minted property is then dropped from the extension entirely). Writes the consolidated
    vocab to CONSOLIDATED_VOCAB_PATH and a report to CONSOLIDATION_REPORT_PATH — the raw
    relation_vocabulary.json is NOT modified (pass load_vocab() as `vocab`)."""
    ents  = list(vocab["properties"])
    by_id = {t["id"]: t for t in ents}

    def _articles_of(t):
        return [p.get("article") for p in t.get("provenance", []) if p.get("article")]
    articles_before = sorted({a for t in ents for a in _articles_of(t)})

    groups = []
    shortlist, _adj = [], {}     # the embedder's proposals, for the audit report
    if len(ents) >= 2:
        ents_sorted = sorted(ents, key=lambda x: x["id"])
        lines = [f'- id: {t["id"]} | label: {t["canonical_label"]} | parent: {t.get("parent")} '
                 f'| support: {t.get("support", 0)} | {_sig(t.get("domain"), t.get("range"))} '
                 f'| def: {t.get("definition", "")}'
                 + (f' | aka: {", ".join(t["aliases"])}' if t.get("aliases") else "")
                 for t in ents_sorted]
        cand_lines, shortlist, _adj = _consolidation_shortlist(ents_sorted)
        gate = ("\nBoth SYNONYM and SUBPROPERTY groups are enabled." if SUBPROP_MODE != "off" else
                "\nReturn only SYNONYM groups; do NOT relate broader/narrower (subproperty) properties.")
        # Each vocab property is followed by its own candidates (vocab and schema.org ranked together),
        # so every id the judge may reach for is defined right where it is offered.
        blocks = [ln + "\n" + "\n".join(cand_lines[t["id"]]) for t, ln in zip(ents_sorted, lines)]
        msg = ("VOCABULARY — each property, then its retrieved CANDIDATES (`->` lines: the only entries "
               "it may be grouped with):\n\n" + "\n\n".join(blocks))
        try:
            res = parse_json(gemini_json(CONSOLIDATE_ALL_PROMPT + gate, msg))
            groups = (res.get("groups", res) if isinstance(res, dict) else res) or []
        except Exception as ex:
            print(f"[WARN] consolidation call failed ({type(ex).__name__}: {ex})")

    def _keep_ref(keep):
        """Resolve a keep_id to ("vocab", entry) or ("schema.org", label); (None, None) if unknown.
        A bare schema.org label (no `schema:` prefix) is tolerated."""
        if not isinstance(keep, str):
            return None, None
        if keep in by_id:
            return "vocab", by_id[keep]
        lbl = keep[len(SCHEMA_ID_PREFIX):] if keep.startswith(SCHEMA_ID_PREFIX) else keep
        if lbl in SCHEMA_LABELS and lbl not in SCHEMA_EXCLUDE_LABELS:   # excluded ones are never valid keeps
            return "schema.org", lbl
        return None, None

    def _split_label_collisions(groups):
        """A schema.org keep that shares its LABEL with one of its children is not a hierarchy — it is
        the same name twice. `parent` stores a bare label, so attaching the child there would make it
        its OWN parent (self-loop: _find_vocab_by_label resolves the label back to the child, and the
        entry drops out of the hierarchy). An identical label means equivalence, so those children are
        rewritten into a SYNONYM merge into schema.org; any remaining children keep the original kind."""
        out = []
        for g in groups:
            ksrc, kobj = _keep_ref(g.get("keep_id"))
            ids  = list(dict.fromkeys(g.get("merge_ids") or []))
            same = [i for i in ids if ksrc == "schema.org" and i in by_id
                    and by_id[i]["canonical_label"] == kobj]
            if not same:
                out.append(g); continue
            note = " [same label as the schema.org entry -> merged, not nested under it]"
            out.append({**g, "kind": "synonym", "merge_ids": same,
                        "reason": (g.get("reason") or "") + note})
            rest = [i for i in ids if i not in same]
            if rest:
                out.append({**g, "merge_ids": rest})
        return out
    groups = _split_label_collisions(groups)

    merges, reparents, merged_ids = [], [], set()
    label_remap = {}          # merged-away label -> survivor label (to repair `parent` pointers)
    for g in groups:
        kind = (g.get("kind") or "synonym").lower()
        is_subprop = kind.startswith("subprop")
        ksrc, kobj = _keep_ref(g.get("keep_id"))
        if ksrc is None or (ksrc == "vocab" and kobj["id"] in merged_ids):
            continue
        # Only VOCAB properties can be merged away — a schema.org id in merge_ids is ignored by design.
        children = [by_id[i] for i in dict.fromkeys(g.get("merge_ids") or [])
                    if i in by_id and i not in merged_ids and not (ksrc == "vocab" and i == kobj["id"])]
        if not children:
            continue

        # SUBPROPERTY + subproperty mode: keep the children, just re-parent them onto the broader property
        if is_subprop and SUBPROP_MODE == "subproperty":
            parent_label = kobj["canonical_label"] if ksrc == "vocab" else kobj
            rec = {"kind": "subproperty", "action": "subproperty",
                   "parent_id": kobj["id"] if ksrc == "vocab" else SCHEMA_ID_PREFIX + kobj,
                   "parent_label": parent_label, "parent_source": ksrc,
                   "reason": g.get("reason"), "children": []}
            for child in children:
                if child.get("parent") == parent_label:
                    continue                                      # already under this parent — no-op
                rec["children"].append({"id": child["id"], "label": child["canonical_label"],
                                        "old_parent": child.get("parent")})
                child["parent"] = parent_label                    # keep the child; re-parent it
            if rec["children"]:
                reparents.append(rec)
            continue

        # otherwise MERGE: synonyms always; subproperty only when SUBPROP_MODE == "merge"
        if is_subprop and SUBPROP_MODE != "merge":
            continue
        if ksrc == "schema.org":
            survivor, members = None, children                    # folds INTO schema.org: nothing survives in vocab
            survivor_id, survivor_label = SCHEMA_ID_PREFIX + kobj, kobj
        elif is_subprop:
            survivor, members = kobj, children                    # broader parent survives
        else:
            members  = [kobj] + children
            survivor = max(members, key=lambda t: (t.get("support", 0), -int(t["id"][1:])))
        if survivor is not None:
            survivor_id, survivor_label = survivor["id"], survivor["canonical_label"]
        rec = {"kind": kind, "action": "merge", "survivor_id": survivor_id,
               "survivor_label": survivor_label,
               "survivor_source": "schema.org" if survivor is None else "vocab",
               "reason": g.get("reason"), "merged": [],
               "articles": sorted({a for t in members for a in _articles_of(t)})}
        for m in members:
            if survivor is not None and m["id"] == survivor["id"]:
                continue
            if survivor is not None:                              # schema.org survivors hold no aliases/provenance
                for lbl in [m["canonical_label"]] + m.get("aliases", []):
                    if lbl != survivor["canonical_label"] and lbl not in survivor["aliases"]:
                        survivor["aliases"].append(lbl)
                survivor["support"] = survivor.get("support", 0) + m.get("support", 0)
                survivor.setdefault("provenance", []).extend(m.get("provenance", []))
            rec["merged"].append({"id": m["id"], "label": m["canonical_label"],
                                  "support": m.get("support", 0), "articles": _articles_of(m)})
            merged_ids.add(m["id"])
            label_remap[m["canonical_label"]] = survivor_label
        if rec["merged"]:
            if survivor is not None:
                rec["resulting_support"]  = survivor["support"]
                rec["resulting_aliases"] = list(survivor["aliases"])
            merges.append(rec)

    if merged_ids:
        vocab["properties"] = [t for t in vocab["properties"] if t["id"] not in merged_ids]
    # A merged-away label can still be some surviving property's `parent` (subPropertyOf) — repoint it
    # at the survivor (following merge chains), so no super-property link dangles after consolidation.
    for t in vocab["properties"]:
        p, seen = t.get("parent"), set()
        while p in label_remap and p not in seen:
            seen.add(p); p = label_remap[p]
        if p != t.get("parent"):
            t["parent"] = p
    # Safety net: an entry must never be its own parent. A bare label can name both a vocab entry and a
    # schema.org one, and the vocab side wins the lookup — so a self-label parent silently orphans the
    # entry from the hierarchy. Drop it to no super-entry and record it in the report.
    self_parented = [t["id"] for t in vocab["properties"] if t.get("parent") == t["canonical_label"]]
    for t in vocab["properties"]:
        if t.get("parent") == t["canonical_label"]:
            t["parent"] = None

    # write the CONSOLIDATED vocabulary to its own file (raw relation_vocabulary.json stays untouched)
    _annotate_vocab_paths(vocab)                 # refresh derived `path`s (re-parenting/merges changed them)
    CONSOLIDATED_VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

    n_reparented  = sum(len(r["children"]) for r in reparents)
    n_into_schema = sum(len(r["merged"]) for r in merges if r["survivor_source"] == "schema.org")
    n_sch_parents = sum(len(r["children"]) for r in reparents if r["parent_source"] == "schema.org")
    # audit: did the embedder's shortlist actually propose each pair the LLM merged/re-parented?
    def _in_shortlist(a, b):
        return b in _adj.get(a, ()) or a in _adj.get(b, ())
    for _r in merges:
        for _m in _r["merged"]:
            _m["in_shortlist"] = _in_shortlist(_r["survivor_id"], _m["id"])
    for _r in reparents:
        for _c in _r["children"]:
            _c["in_shortlist"] = _in_shortlist(_r["parent_id"], _c["id"])

    report = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "match_model": MATCH_MODEL,
        "gemini_model": GEMINI_MODEL, "expand_mode": EXPAND_MODE,
        "subprop_mode": SUBPROP_MODE, "source_vocab": str(VOCAB_PATH.name),
        "consolidated_vocab": str(CONSOLIDATED_VOCAB_PATH.name),
        "consolidate_top_k": CONSOLIDATE_TOP_K,
        "candidate_pool": ("vocab+schema.org" if CONSOLIDATE_INCLUDE_SCHEMA else "vocab"),
        "schema_excluded_labels": sorted(SCHEMA_EXCLUDE_LABELS), "self_parent_fixed": self_parented,
        "neighbour_shortlist": shortlist,
        "articles_used": articles_before, "n_articles": len(articles_before),
        "vocab_size_before": len(ents), "vocab_size_after": len(vocab["properties"]),
        "n_merges": len(merges), "n_props_removed": len(merged_ids), "n_reparented": n_reparented,
        "n_merged_into_schema": n_into_schema, "n_reparented_under_schema": n_sch_parents,
        "merges": merges, "reparents": reparents,
    }
    if write_report:
        CONSOLIDATION_REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False))

    if verbose:
        print(f"consolidation (1 call): subprop_mode={SUBPROP_MODE!r} | pool={report['candidate_pool']} | "
              f"{len(merges)} merge-group(s), {len(merged_ids)} removed "
              f"({n_into_schema} into schema.org), {n_reparented} re-parented "
              f"({n_sch_parents} under schema.org) | vocab {len(ents)} -> {len(vocab['properties'])}")
        for r in merges:
            tag = "schema.org" if r["survivor_source"] == "schema.org" else r["survivor_id"]
            print(f"  [merge/{r['kind'][:4]}] keep {tag} {r['survivor_label']!r} <= "
                  f"{[m['label'] for m in r['merged']]}")
        for r in reparents:
            tag = " (schema.org)" if r["parent_source"] == "schema.org" else ""
            print(f"  [subproperty] {[c['label'] for c in r['children']]} subPropertyOf {r['parent_label']}{tag}")
        print(f"  consolidated vocab -> {CONSOLIDATED_VOCAB_PATH}  (raw {VOCAB_PATH.name} untouched)")
        if write_report:
            print(f"  report -> {CONSOLIDATION_REPORT_PATH}")
    return report

# Runs on the RAW incremental vocab and writes the consolidated result to a separate file.
_ = consolidate_vocab(load_vocab())

consolidation (1 call): subprop_mode='subproperty' | pool=vocab+schema.org | 4 merge-group(s), 5 removed (3 into schema.org), 25 re-parented (10 under schema.org) | vocab 81 -> 76
  [merge/syno] keep P0047 'participatedInEvent' <= ['participatesInEvent']
  [merge/syno] keep schema.org 'isRelatedTo' <= ['associatedWith']
  [merge/syno] keep schema.org 'publishedBy' <= ['declaredBy', 'expressedBy']
  [merge/syno] keep P0061 'infectsTaxon' <= ['spreadsToTaxon']
  [subproperty] ['recordDate'] subPropertyOf observationDate (schema.org)
  [subproperty] ['hasAlert'] subPropertyOf hasStatus
  [subproperty] ['fromQuantity', 'toQuantity', 'hasPopulation'] subPropertyOf hasQuantity
  [subproperty] ['sourceCountry'] subPropertyOf countryOfOrigin (schema.org)
  [subproperty] ['warmingRate', 'increasedByPercentage'] subPropertyOf increases
  [subproperty] ['expectedAttendeeCount'] subPropertyOf futureState
  [subproperty] ['activeOnDate', 'activeAtTime'] subPropertyOf presentDuring
  [subproperty] [

### Refresh `nary_graph_extended.json` after consolidation → **separate** files — no LLM

Apply the consolidation's `merged_id → survivor` mapping (and subproperty re-parentings) to the
per-article files deterministically, writing the result to a **new** `nary_graph_extended_consolidated.json`
per article — the incremental `nary_graph_extended.json` is left untouched. When a property merged
**into schema.org**, the record stops being an extension property: `property_source` flips to
`"schema.org"` and `vocab_id` / `property_definition` / `property_parent` go null, exactly like a
record the build had labelled with schema.org directly. Also emit one global
`relation_consolidation_ext_changes.json` listing **only** the records that changed (before → after).

In [33]:
# Produce a CONSOLIDATED copy of each per-article nary_graph_extended.json WITHOUT the LLM — the incremental
# nary_graph_extended.json files are left UNTOUCHED. Three effects, applied only to the copy:
#   - MERGED properties: records pointing at a removed vocab_id are remapped to its survivor.
#   - MERGED INTO SCHEMA.ORG: when the survivor is a schema.org property the record stops being an
#     extension property — property_source becomes "schema.org", vocab_id/definition/parent null.
#   - RE-PARENTED properties (subproperty mode): the property survives, so we re-sync every vocab-typed
#     record's label / definition / parent from the CONSOLIDATED vocabulary (picks up the new parent).
# `property_is_new_to_vocab` is a BUILD-TIME provenance fact — consolidation leaves it untouched.
# Writes, per article, nary_graph_extended_consolidated.json, and ONE global changes file listing ONLY the
# records that changed (before -> after). Pure bookkeeping. Run right after consolidate_vocab.
EXT_CONSOLIDATED_NAME = f"nary_graph{_osfx}_extended_consolidated.json"                       # per-article refreshed copy
EXT_CHANGES_PATH      = RESULTS_DIR / f"_global/e2e/nary_relation_consolidation_ext_changes{_osfx}.json" # global "what changed" file
_CHANGED_FIELDS = ("vocab_id", "property_source", "property", "property_definition", "property_parent")  # substantive fields

def refresh_ext_after_consolidation(report=None, verbose=True):
    if report is None:
        if not CONSOLIDATION_REPORT_PATH.exists():
            print("no consolidation report found — run consolidate_vocab first"); return
        report = json.loads(CONSOLIDATION_REPORT_PATH.read_text())
    # merged vocab id -> where it went: another vocab id, or a schema.org property label
    remap = {}
    for r in report.get("merges", []):
        tgt = ({"source": "schema.org", "label": r["survivor_label"]}
               if r.get("survivor_source") == "schema.org" else
               {"source": "vocab", "id": r["survivor_id"]})
        for m in r.get("merged", []):
            remap[m["id"]] = tgt
    by_id = {t["id"]: t for t in load_consolidated()["properties"]}   # survivors + re-parented properties

    def _resolve(v):
        """The consolidated fields for a record currently using vocab id `v` (None: leave as-is).
        Follows merge chains (a survivor of one group can be merged again by a later one)."""
        tgt, seen = remap.get(v), set()
        while tgt is not None and tgt["source"] == "vocab" and tgt["id"] in remap and tgt["id"] not in seen:
            seen.add(tgt["id"]); tgt = remap[tgt["id"]]
        if tgt is not None and tgt["source"] == "schema.org":         # left the extension for schema.org
            return {"vocab_id": None, "property_source": "schema.org", "property": tgt["label"],
                    "property_definition": None, "property_parent": None}
        s = by_id.get(tgt["id"] if tgt else v)                        # follow a merge, else same id
        if s is None:
            return None
        return {"vocab_id": s["id"], "property_source": "vocab", "property": s["canonical_label"],
                "property_definition": s.get("definition"), "property_parent": s.get("parent")}

    all_changes, total = [], 0
    for fp in sorted(RESULTS_DIR.glob(f"*/e2e/nary_graph{_osfx}_extended.json")):
        data = json.loads(fp.read_text())                            # the untouched incremental file
        aid  = data.get("article_id", fp.parent.name)
        art_changes = []
        for n in data["relations"]:
            v = n.get("vocab_id")
            if not v:
                continue
            new = _resolve(v)
            if new is None:
                continue
            old = {k: n.get(k) for k in _CHANGED_FIELDS}
            if new != old:
                n.update(new)                                        # mutate the in-memory COPY only
                art_changes.append({"subject_text": n.get("subject_text"), "object_text": n.get("object_text"),
                                    "before": old, "after": new})
        data["consolidated"], data["subprop_mode"] = True, SUBPROP_MODE
        (fp.parent / EXT_CONSOLIDATED_NAME).write_text(json.dumps(data, indent=2, ensure_ascii=False))
        if art_changes:
            all_changes.append({"article_id": aid, "n_changed": len(art_changes), "changes": art_changes})
            total += len(art_changes)
            if verbose:
                print(f"  {aid}: {len(art_changes)} record(s) changed")

    EXT_CHANGES_PATH.write_text(json.dumps({
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "subprop_mode": SUBPROP_MODE,
        "consolidated_vocab": str(CONSOLIDATED_VOCAB_PATH.name),
        "n_articles_changed": len(all_changes), "n_records_changed": total, "articles": all_changes,
    }, indent=2, ensure_ascii=False))
    print(f"done — wrote {EXT_CONSOLIDATED_NAME} for each article (incremental nary_graph_extended.json untouched)\n"
          f"       {total} changed record(s) across {len(all_changes)} article(s) -> {EXT_CHANGES_PATH.name}")

refresh_ext_after_consolidation()

  art1: 5 record(s) changed
  art2: 15 record(s) changed
  art4: 14 record(s) changed
  art5: 23 record(s) changed
done — wrote nary_graph_dedup_rr_extended_consolidated.json for each article (incremental nary_graph_extended.json untouched)
       57 changed record(s) across 4 article(s) -> nary_relation_consolidation_ext_changes_dedup_rr.json


## Property hierarchy (schema.org + extension) — raw and consolidated, **separate** files

One indented tree of the property space — schema.org properties (linked by `subPropertyOf`) and the
extension (minted) properties together, each tagged `[schema.org]` or `[extension]`, sub-properties
nested under their parent. Built **twice**: `property_hierarchy_ext.txt` from the **raw incremental**
vocab, and `property_hierarchy_ext_consolidated.txt` from the **consolidated** vocab. Set
`INCLUDE_ALL_SCHEMA=True` to include every schema.org property, or leave it `False` to show just the
branches your extension properties hang off (extension properties + their super-property chains).

In [34]:
# Property hierarchy: schema.org properties + extension (vocab) properties in ONE indented tree, each
# tagged [schema.org] or [extension]; sub-properties nested under their subPropertyOf parent.
#   property_hierarchy_ext.txt              <- raw incremental vocab (relation_vocabulary.json)
#   property_hierarchy_ext_consolidated.txt <- consolidated vocab   (re-parentings/merges reflected)
#   INCLUDE_ALL_SCHEMA=False -> only extension properties + their super-property chains.
#   INCLUDE_ALL_SCHEMA=True  -> the entire schema.org property set too (large).
INCLUDE_ALL_SCHEMA               = False
HIERARCHY_PATH                   = RESULTS_DIR / f"_global/e2e/property_hierarchy_ext{_osfx}.txt"                # raw incremental
HIERARCHY_CONSOLIDATED_PATH      = RESULTS_DIR / f"_global/e2e/property_hierarchy_ext{_osfx}_consolidated.txt"   # consolidated

def build_property_hierarchy(vocab, write_path, include_all_schema=INCLUDE_ALL_SCHEMA, show=50):
    schema_labels = {p["label"] for p in schema_pool}
    schema_parent = {p["label"]: _schema_super(p["label"]) for p in schema_pool}
    ext = {t["canonical_label"]: t for t in vocab["properties"]}

    parent_of, kind = {}, {}
    for lbl in schema_labels:
        parent_of[lbl] = schema_parent.get(lbl); kind[lbl] = "schema.org"
    for lbl, t in ext.items():                                   # extension overrides if same label
        parent_of[lbl] = t.get("parent"); kind[lbl] = "extension"

    if include_all_schema:
        included = set(schema_labels) | set(ext)
    else:                                                        # only extensions + their ancestor chains
        included = set(ext)
        for lbl in list(ext):
            cur, seen = parent_of.get(lbl), set()
            while cur and cur not in seen and cur not in included:
                seen.add(cur); included.add(cur); cur = parent_of.get(cur)

    children, roots = defaultdict(list), []
    for lbl in included:
        p = parent_of.get(lbl)
        (children[p].append(lbl) if p in included else roots.append(lbl))
    for p in children:
        children[p].sort()

    lines = []
    def walk(lbl, depth, seen):
        tag   = "[extension]" if kind.get(lbl) == "extension" else "[schema.org]"
        extra = f"  (support {ext[lbl].get('support', 0)})" if lbl in ext else ""
        lines.append("  " * depth + f"{lbl}  {tag}{extra}")
        for c in children.get(lbl, []):
            if c not in seen:
                walk(c, depth + 1, seen | {c})
    for r in sorted(set(roots)):
        walk(r, 0, {r})

    text = "\n".join(lines)
    if write_path:
        write_path.write_text(text)
    print(f"property hierarchy: {len(lines)} nodes ({len(ext)} extension) | "
          f"include_all_schema={include_all_schema} -> {write_path}\n" + "-" * 74)
    print(text if len(lines) <= show else "\n".join(lines[:show]) + f"\n  ... ({len(lines) - show} more — see file)")
    return text

# Raw incremental hierarchy (always available), then the consolidated one (reflects re-parentings/merges).
print("=== RAW incremental property vocabulary ===")
build_property_hierarchy(load_vocab(),        HIERARCHY_PATH)
print("\n=== CONSOLIDATED property vocabulary ===")
build_property_hierarchy(load_consolidated(), HIERARCHY_CONSOLIDATED_PATH)

=== RAW incremental property vocabulary ===
property hierarchy: 102 nodes (81 extension) | include_all_schema=False -> /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/gemini-2.5-flash/_global/e2e/property_hierarchy_ext_dedup_rr.txt
--------------------------------------------------------------------------
about  [schema.org]
  isFor  [extension]  (support 3)
  reportsAbout  [extension]  (support 8)
activeAtTime  [extension]  (support 2)
activeOnDate  [extension]  (support 2)
admires  [extension]  (support 2)
agent  [schema.org]
  declaredBy  [extension]  (support 1)
areaServed  [schema.org]
  eligibleRegion  [schema.org]
    restrictsToRegion  [extension]  (support 3)
award  [schema.org]
  awardedMonetaryAmount  [extension]  (support 6)
  receivesPrizeMoney  [extension]  (support 2)
cause  [schema.org]
  causedByEvent  [extension]  (support 1)
causeOf  [schema.org]
  causesHarmTo  [extension]  (support 2)
comparedTo  [extension]  (support 3)
co

'about  [schema.org]\n  isFor  [extension]  (support 3)\n  reportsAbout  [extension]  (support 8)\n    reportsParticipant  [extension]  (support 1)\nadmires  [extension]  (support 2)\naward  [schema.org]\n  awardedMonetaryAmount  [extension]  (support 6)\ncauseOf  [schema.org]\n  causesHarmTo  [extension]  (support 2)\ncomparedTo  [extension]  (support 3)\nconfirmsAlternativeEntry  [extension]  (support 1)\nconfirmsEntryUser  [extension]  (support 3)\nconfirmsPurpose  [extension]  (support 1)\ncountryOfOrigin  [schema.org]\n  sourceCountry  [extension]  (support 3)\ncoverageEndTime  [schema.org]\n  reportsExpectedEndTime  [extension]  (support 1)\ncoverageStartTime  [schema.org]\n  reportsExpectedStartTime  [extension]  (support 1)\ncreator  [schema.org]\n  informationSource  [extension]  (support 14)\ndatePublished  [schema.org]\n  publishedOnDate  [extension]  (support 1)\ndefeated  [extension]  (support 1)\ndiagnosedWith  [extension]  (support 2)\nerror  [schema.org]\n  hindersEvent